# P&ID Stage 2 — encoder one-shot toàn cục trên Colab (GPU)

Train **một** encoder ResNet-34 + ArcFace trên crop DigitizePID (`fgc.global_encoder=true`),
rồi dựng prototype từ template ISO 10628-2. Thứ tự (docs/PLAN_STAGE2_ONESHOT.md, Phase B;
hướng dẫn từng bước: docs/COLAB_RUNBOOK.md §5c):

| Bước | Mục | Ra file trên Drive (`runs/encoders/`) |
|---|---|---|
| R1b — encoder đủ 32 class (lõi luận văn) — **chạy TRƯỚC** | 6 | `__global__.pt` |
| R1a — encoder **leave-out** (protocol P1) | 5 | `__global_lo__.pt` |
| R2 — prototype ISO + separation | 7 | `iso_prototypes.pt`, `iso_prototypes_lo.pt` |
| B4a — baseline chỉ-ImageNet (không train) | 10 | `__imagenet__.pt`, `iso_prototypes_imagenet.pt` |

Chạy theo thứ tự mục: 1 → 2 → 3 → 4 → **6** → 5 → 7 (mục 6 nằm sau mục 5 trong file).
R1b đã xong (2026-11-12): phiên R1a chạy 1 → 2 → 3 → 4 → 5 → 7.

**Trước khi chạy**: nhánh đã **push** lên GitHub (Colab clone code từ đó); danh sách H đã chốt trong
`configs/config.yaml` (`oneshot.holdout`, 2026-11-11). Mục 4 dừng nếu danh sách trống.
**Runtime**: GPU (T4 đủ). Crop + augment chạy trên CPU (`LOADER` ở mục 4).

## 1. GPU + mount Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
# Thư mục dự án trên Drive phải chứa data/canonical/. Notebook Stage 1 dùng MyDrive/pid, bản
# Stage 2 cũ dùng MyDrive/PID_Recognition -> thử cả hai; sửa DRIVE_ROOT nếu ở chỗ khác.
DRIVE_ROOT = next((d for d in ('/content/drive/MyDrive/pid', '/content/drive/MyDrive/PID_Recognition')
                   if os.path.isdir(f'{d}/data/canonical')), None)
assert DRIVE_ROOT, 'Không thấy <DRIVE_ROOT>/data/canonical — sửa DRIVE_ROOT bằng tay'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_DIR  = f'{DRIVE_ROOT}/runs'
ENC_DIR   = f'{RUNS_DIR}/encoders'      # encoder + prototype đã đặt tên rõ ràng
os.makedirs(ENC_DIR, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('canonical images:', len(os.listdir(f'{DATA_ROOT}/canonical/images')))

## 2. Code + deps

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'   # <-- sửa
BRANCH   = 'feat/prepare-for-colab-stage-2'
%cd /content
![ -d pid ] || git clone -b $BRANCH $REPO_URL pid
%cd /content/pid
!git fetch -q && git checkout -q $BRANCH && git pull -q && git log --oneline -1
!pip install -q uv
# Colab đã có torch CUDA -> không cài torch. run_pipeline.py import mọi stage (cả detector),
# nên cần đủ ultralytics/sahi/ensemble-boxes dù Stage 2 không dùng tới.
!uv pip install --system -q ultralytics sahi ensemble-boxes albumentations timm hydra-core omegaconf \
    pycocotools opencv-python-headless pillow pandas scikit-learn matplotlib tqdm pyyaml
import sys
os.environ['PYTHONPATH'] = '/content/pid/src'
sys.path.insert(0, '/content/pid/src')
REPO_DATA = '/content/pid/data/raw'     # template ISO + vocab.csv nằm trong git, không ở Drive

## 3. Crop toàn cục — đủ 32 class

**Bẫy im lặng**: `derived/crops_fgc/` và `crops_fgc.zip` cũ trên Drive chỉ có **28 class** (cắt theo
`fgc_group`). Notebook này KHÔNG BAO GIỜ đọc chúng: crop toàn cục được cắt ở local rồi lưu thành
`crops_fgc_ctx.zip` (tên khác). Cắt vào thư mục cũ cũng không được — tên file đánh số khác
nhau giữa hai chế độ nên sẽ ra crop trùng.

**Định dạng crop mới (2026-11-11, `utils/crop_box.py`)**: mỗi PNG là vùng ngữ cảnh quanh ký hiệu, kèm
`<lớp>/_boxes.csv` (vị trí box). Dataset cắt ra view box + 10% (giống lúc suy luận), hoặc view lệch
ngẫu nhiên khi train (`box_jitter`). Zip cũ `crops_fgc_global.zip` (không có `_boxes.csv`) **không dùng**
nữa — cell dưới kiểm `_boxes.csv` ở mọi lớp.

Phiên đầu: copy ảnh canonical (~1.2 GB) về local, cắt, zip lên Drive. Phiên sau: chỉ giải nén.

In [ ]:
import glob, shutil, time
from utils.classmap import load_classmap
LOCAL_DATA_ROOT = '/content/data'
LOCAL_CROPS = f'{LOCAL_DATA_ROOT}/derived/crops_fgc'
GLOBAL_ZIP  = f'{DATA_ROOT}/derived/crops_fgc_ctx.zip'
t0 = time.time()
os.makedirs(f'{LOCAL_DATA_ROOT}/canonical', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_ROOT}/derived', exist_ok=True)
for f in ('classes.yaml', 'manifest.csv', 'annotations.coco.json', 'digitizepid_splits.json'):
    if os.path.exists(f'{DATA_ROOT}/canonical/{f}'):
        shutil.copy2(f'{DATA_ROOT}/canonical/{f}', f'{LOCAL_DATA_ROOT}/canonical/{f}')

if os.path.exists(GLOBAL_ZIP):
    !unzip -q -o "$GLOBAL_ZIP" -d "$LOCAL_DATA_ROOT/derived/"
else:
    !rsync -a "$DATA_ROOT/canonical/images/" "$LOCAL_DATA_ROOT/canonical/images/"
    for step, f in (('classes', 'classes.yaml'), ('manifest', 'manifest.csv')):
        if not os.path.exists(f'{LOCAL_DATA_ROOT}/canonical/{f}'):
            !cd /content/pid && python src/run_pipeline.py prep --step $step data_root=$LOCAL_DATA_ROOT
    !cd /content/pid && python src/run_pipeline.py prep --step crops data_root=$LOCAL_DATA_ROOT fgc.global_encoder=true
    !cd "$LOCAL_DATA_ROOT/derived" && zip -q -r crops_fgc_ctx.zip crops_fgc && cp crops_fgc_ctx.zip "$GLOBAL_ZIP"
    print('Đã lưu', GLOBAL_ZIP)
print(f'Crops local sau {time.time()-t0:.0f}s')

# Phải đủ ĐÚNG mọi fine class của classes.yaml ở cả hai split (32 với DigitizePID).
CM = load_classmap(f'{LOCAL_DATA_ROOT}/canonical/classes.yaml')
for split in ('train', 'val'):
    present = set(os.listdir(f'{LOCAL_CROPS}/{split}'))
    n = len(glob.glob(f'{LOCAL_CROPS}/{split}/*/*.png'))
    print(f'  {split}: {n} crops, {len(present)} fine classes')
    missing = sorted(set(CM.fine_classes) - present)
    assert not missing, f'{split} thiếu class {missing} — zip cũ 28 class? xoá {GLOBAL_ZIP} rồi chạy lại'
    assert len(present) == len(CM.fine_classes) == 32, sorted(present)
    no_boxes = sorted(c for c in present if not os.path.exists(f'{LOCAL_CROPS}/{split}/{c}/_boxes.csv'))
    assert not no_boxes, f'{split}: thiếu _boxes.csv ở {no_boxes} — crop định dạng cũ? xoá {GLOBAL_ZIP} rồi chạy lại'

## 4. Danh sách giữ ra H (từ config, KHÔNG gõ tay)

Đọc `oneshot.holdout` trong `configs/config.yaml` (chốt 2026-11-11). R1a train với
`fgc.exclude_classes = h_in + h_out + h_extra` (`h_extra`: VARIANT cùng ký hiệu ISO với một lớp
`h_in`, bỏ khỏi train để không rò rỉ, không thi). Tên sai -> lỗi ngay tại đây, trước khi tốn GPU.

Cell này cũng đặt `LOADER` (2 worker nạp dữ liệu song song) và `AUG` (augment cho **cả R1a và R1b**, kể cả khi resume): suy giảm kiểu scan +
lệch box (PLAN, quyết định 2026-11-11). H phải chốt trước **cả hai** lượt train, nên R1b cũng cần
cell này. **R1b chạy trước R1a** (mục 6 trước mục 5).

In [ ]:
from omegaconf import OmegaConf
from utils.classmap import GLOBAL_GROUP, training_fine_classes
from utils.oneshot import holdout_classes
CFG = OmegaConf.load('/content/pid/configs/config.yaml')
H_IN, H_OUT, H_EXTRA = holdout_classes(CFG)             # trùng lặp giữa các list -> ValueError
H = H_IN + H_OUT                                         # lớp được thi ở P1 (H_EXTRA: bỏ khỏi train, KHÔNG thi)
assert H, 'oneshot.holdout trống: làm B5 (chọn H_in/H_out), sửa configs/config.yaml, push, rồi chạy lại cell 2'
HELD = H + H_EXTRA                                       # mọi lớp R1a không được thấy
KEPT = training_fine_classes(CM, GLOBAL_GROUP, HELD)    # tên không có trong classes.yaml -> ValueError
EXCLUDE = '[' + ','.join(HELD) + ']'
print(f'H_in    ({len(H_IN)}):', H_IN)
print(f'H_out   ({len(H_OUT)}):', H_OUT)
print(f'h_extra ({len(H_EXTRA)}):', H_EXTRA)
print(f'R1a train trên {len(KEPT)}/{len(CM.fine_classes)} class')

# Augment cho R1a VÀ R1b, kể cả resume (PLAN 2026-11-11). '++' thêm hoặc ghi đè key.
AUG = '++fgc.aug.degrade.p=0.5 ++fgc.aug.box_jitter.p=0.5'
# Augment chạy trên CPU; 2 worker = số nhân CPU của Colab thường. 0 = như cũ (chậm hơn).
LOADER = '++fgc.workers=2'
print('AUG:', AUG, '| LOADER:', LOADER)

# Quy tắc checkpoint (PLAN, chốt 2026-11-13): val_acc bão hoà từ epoch 1 nên "best val_acc" chọn
# epoch ngẫu nhiên. R1a và R1b so ở CÙNG epoch CMP_EPOCH. fgc_train ghi __global__.ep<N>.pt ở mỗi
# epoch trong save_epochs, và __global__.last.pt mỗi epoch (resume tiếp từ đó).
CMP_EPOCH = 3
EPOCHS = 5
SAVE = '++fgc.save_epochs=[3,5]'   # trong lệnh: "$SAVE" có ngoặc kép (dấu [ ] là glob của shell)
print('EPOCHS:', EPOCHS, '| SAVE:', SAVE, '| so ở epoch', CMP_EPOCH)

**4b. Hàm `finalize`** — chạy một lần mỗi phiên, trước 5b / 6b. Lấy checkpoint epoch `CMP_EPOCH`
của run, kiểm encoder không có cột nào của các lớp bị giữ ra, rồi copy vào `encoders/`.

In [ ]:
import torch
def finalize(run, out_name, expect_excluded, epoch=CMP_EPOCH):
    # Mọi phiên của run (kể cả phiên resume), .ep<N>.pt trước. Run trước 2026-11-13 không có
    # __global__.ep<N>.pt: khi đó dùng __global__.pt nếu nó đúng là epoch đó.
    newest = lambda pat: sorted(glob.glob(f'{RUNS_DIR}/{run}/*/fgc_checkpoints/{pat}'), key=os.path.getmtime, reverse=True)
    for ck in newest(f'__global__.ep{epoch}.pt') + newest('__global__.pt'):
        st = torch.load(ck, map_location='cpu', weights_only=False)
        if st['epoch'] + 1 == epoch:
            break
    else:
        raise FileNotFoundError(f'{RUNS_DIR}/{run}: không có checkpoint epoch {epoch} — train chưa tới epoch {epoch}?')
    fine = list(st['fine_classes'])
    assert not set(expect_excluded) & set(fine), f'RÒ RỈ: {sorted(set(expect_excluded) & set(fine))} có trong encoder'
    assert len(fine) == len(CM.fine_classes) - len(expect_excluded), (len(fine), len(expect_excluded))
    assert sorted(st.get('excluded_classes', [])) == sorted(expect_excluded), st.get('excluded_classes')
    assert st['model']['head.weight'].shape[0] == len(fine)
    shutil.copy2(ck, f'{ENC_DIR}/{out_name}')
    print(f'{ck}\n  -> {ENC_DIR}/{out_name}: {len(fine)} class, epoch {st["epoch"]+1}, val_acc {st["val_acc"]:.4f}')

## 5. R1a — encoder leave-out → `__global_lo__.pt`

Train `EPOCHS` epoch (mục 4). Checkpoint ghi thẳng vào Drive, `runs/r1a/<ts>/fgc_checkpoints/`:
`__global__.ep3.pt`, `__global__.ep5.pt` (epoch cố định), `__global__.last.pt` (mỗi epoch, để resume),
`__global__.pt` (best val_acc, KHÔNG dùng để so). Cell 5b lấy epoch `CMP_EPOCH` → `encoders/__global_lo__.pt`.

In [ ]:
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1a \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=$EXCLUDE"

**5a. Resume R1a** (nếu Colab ngắt) — tự lấy run R1a mới nhất; CÙNG danh sách H (code chặn nếu khác).

In [ ]:
# Phiên mới nhất có checkpoint; fgc_train tự lấy __global__.last.pt trong đó (cũ: __global__.pt).
R1A_LAST = sorted(glob.glob(f'{RUNS_DIR}/r1a/*/fgc_checkpoints/__global__*.pt'), key=os.path.getmtime)[-1]
R1A_RESUME = os.path.dirname(R1A_LAST)
print('resume từ', R1A_RESUME)
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1a \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=$EXCLUDE" fgc.resume_from=$R1A_RESUME

**5b. Kiểm + đặt tên** — encoder KHÔNG được có cột nào của H hay `h_extra`. Cần cell 4b.

In [ ]:
finalize('r1a', '__global_lo__.pt', HELD)  # h_in + h_out + h_extra, epoch CMP_EPOCH

## 6. R1b — encoder đủ 32 class → `__global__.pt`

Lõi luận văn (protocol C), P2, P4 và end-to-end. **Chạy trước R1a** (PLAN 2026-11-09). `fgc.exclude_classes=[]` ghi rõ để không thừa hưởng nhầm.

**Đã chạy 2026-11-12** (Colab ngắt sau epoch 3; `encoders/__global__.pt` = epoch 3 = `CMP_EPOCH`).
Không chạy lại mục 6 trừ khi đổi cách train (vd. chia B, nhãn theo nghĩa ISO).

In [ ]:
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1b \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=[]"

**6a. Resume R1b** (nếu Colab ngắt).

In [ ]:
# Phiên mới nhất có checkpoint; fgc_train tự lấy __global__.last.pt trong đó (cũ: __global__.pt).
R1B_LAST = sorted(glob.glob(f'{RUNS_DIR}/r1b/*/fgc_checkpoints/__global__*.pt'), key=os.path.getmtime)[-1]
R1B_RESUME = os.path.dirname(R1B_LAST)
print('resume từ', R1B_RESUME)
!cd /content/pid && python src/run_pipeline.py fgc --train \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/r1b \
    fgc.global_encoder=true fgc.metric_learning=true fgc.epochs=$EPOCHS fgc.batch=128 $AUG $LOADER "$SAVE" \
    "fgc.exclude_classes=[]" fgc.resume_from=$R1B_RESUME

**6b. Kiểm + đặt tên.** Cần cell 4b.

In [ ]:
finalize('r1b', '__global__.pt', [])  # epoch CMP_EPOCH

## 7. R2 — prototype ISO + separation (GĐ2)

Mỗi encoder có sẵn trong `encoders/` -> một prototype directory. Đọc dòng
`prototype separation: mean off-diagonal cos=…, max=… (closest pair: A vs B)` và ghi số vào HANDOFF
(bảng B4 trong plan: max < ~0.8 tốt; mean ≥ ~0.6 = encoder không transfer -> B9 prototype có stub,
rồi BACKUP_PLAN; GĐ3 đã bỏ).

In [ ]:
for enc, out in (('__global_lo__.pt', 'iso_prototypes_lo.pt'), ('__global__.pt', 'iso_prototypes.pt')):
    if not os.path.exists(f'{ENC_DIR}/{enc}'):
        print('bỏ qua (chưa có)', enc); continue
    print(f'===== {enc} -> {out}')
    !cd /content/pid && python src/run_pipeline.py prototypes run_dir=$RUNS_DIR/r2 \
        oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$out \
        oneshot.symbols_dir=$REPO_DATA/iso_symbols_clean oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv \
        2>&1 | grep -v "^Warning"

## 8. Eval closed-set (protocol C) — `__global__.pt` trên crop val, nửa `test`

Chạy sau mục 6b. `eval.mode=fgc_only` chấm head 32 lớp của R1b trên crop GT, **không cần detector**:
top-1/top-3, macro-F1, P/R/F1 từng lớp, confusion 32×32 (`confusion___global__.png`), tỉ lệ lỗi
nhảy sang họ coarse khác. Ra `runs/eval_c/<ts>/metrics.json` (PLAN B6c).

End-to-end (mAP 32 lớp fine, cạnh YOLO một-stage) cần thêm Stage 1: `detect` → `fgc`
(`fgc.global_encoder=true fgc.ckpt_dir=$ENC_DIR`) → `evaluate eval.mode=det_plus_fgc`; YOLO 32 lớp
chấm bằng `eval.mode=det_only detector.label_level=fine`. `++eval.image_subset=test`: chỉ 50 ảnh `test` của val (`configs/splits/digitizepid_val_test.json`), nửa còn lại `val_sel`
dành cho chọn τ (§9). Eval one-shot P1/P2: §9; P4 (OPEN100-ISO): §11.

In [ ]:
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_c eval.mode=fgc_only eval.split=val \
    ++eval.image_subset=test fgc.global_encoder=true fgc.metric_learning=true fgc.ckpt_dir=$ENC_DIR
import json
M_PATH = sorted(glob.glob(f'{RUNS_DIR}/eval_c/*/metrics.json'), key=os.path.getmtime)[-1]
m = json.load(open(M_PATH))['fgc']
print(M_PATH)
print(f"{m['n_samples']} crop, {m['n_classes']} lớp: top1 {m['top1']:.4f}  top3 {m['top3']:.4f}  "
      f"macro-F1 {m['macro_f1']:.4f}  nhảy họ {m['cross_family_error_rate']:.4f}")
for name, r in sorted(m['per_class'].items(), key=lambda kv: kv[1]['f1'])[:5]:  # 5 lớp yếu nhất
    print(f"  {name:10s} P {r['precision']:.3f}  R {r['recall']:.3f}  F1 {r['f1']:.3f}  n={r['support']}")

## 9. Eval one-shot mức crop (B6b-1) — chọn τ, P1, P2

Cần mục 7 (prototype) và mục 4 (`oneshot.holdout`). `eval.mode=oneshot` nhúng mỗi crop GT (view box + 10%) bằng
`oneshot.query_views`, gộp bằng `oneshot.aggregate=mean`, so với **mọi** prototype (không route theo coarse), không gate.
Mỗi lượt ghi `runs/eval_*/<ts>/metrics.json` (khóa `oneshot`) + `oneshot_<protocol>_crops.csv` (top-3 tên + cosine từng
crop, phân tích lại không cần GPU) + `oneshot_<protocol>_sweep.png`.

Thứ tự bắt buộc (PLAN B8): **9a chọn τ trên `val_sel`** trước, rồi 9b/9c mới chạm `test`. Hai cell 9a chỉ nhìn lớp
nhóm S mà encoder đã train; **không** ghi τ vào config. 9b/9c tự đọc τ của 9a và báo "Other" tại cả ba τ (`rule_a`, `rule_b`,
`rule_b_no_twins`) từ CSV; `other_at_tau` trong metrics.json là tại `oneshot.other_min_cosine` (placeholder 0.35), bỏ qua.

In [ ]:
# τ ở 9b/9c đọc lại từ metrics.json của cell này (không cần biến sống qua phiên).
import csv as _csv
import numpy as np
from utils.oneshot_metrics import other_prf
from utils.oneshot_metrics import gated_other_prf, margins
def _tau_metrics(tag):
    return json.load(open(sorted(glob.glob(f'{RUNS_DIR}/eval_tau_{tag}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
def chosen_taus(tag):
    o = _tau_metrics(tag)
    return {r: o[r]['tau'] for r in ('rule_a', 'rule_b', 'rule_b_no_twins')}
def chosen_delta(tag):
    # δ của lề top1−top2 (dạng điểm 2), chọn cùng lượt τ trên val_sel; None nếu lượt τ chạy trước khi có δ.
    return _tau_metrics(tag).get('margin_rule_a', {}).get('delta')
def at_chosen_taus(o, tag, known_groups, unknown_groups, keep=None, open_set=None):
    # P/R/F1 của "Other" + tỉ lệ crop biết được nhận VÀ gọi đúng, tại τ của 9a (PLAN B8: báo cả hai quy tắc),
    # rồi cổng "cos < τ_A HOẶC lề < δ" (dạng điểm 2, ablation). keep / open_set: lọc dòng CSV + khối AUROC (P4: một
    # phiên bản ảnh, §11).
    rows = [r for r in _csv.DictReader(open(o['crops_csv'])) if keep is None or keep(r)]
    known = [r for r in rows if r['group'] in known_groups]
    unknown = [r for r in rows if r['group'] in unknown_groups]
    kc = [float(r['best_cos']) for r in known]
    uc = [float(r['best_cos']) for r in unknown]
    for rule, t in chosen_taus(tag).items():
        ccr = np.mean([float(r['best_cos']) >= t and r['top1'] == r['expected'] for r in known])
        prf = other_prf(kc, uc, t) or {}
        print(f"  {rule:16s} τ={t:.4f}  biết: nhận+đúng {ccr:.4f}  |  Other: P {prf.get('precision')}  R {prf.get('recall')}  F1 {prf.get('f1')}")
    d, t = chosen_delta(tag), chosen_taus(tag)['rule_a']
    if d is None:
        print('  (chưa có δ: chạy lại 9a sau khi pull code có lề)'); return
    gate = lambda rs: (np.array([float(r['best_cos']) for r in rs]) < t) | (
        margins([float(r['best_cos']) for r in rs], [float(r['second_cos']) for r in rs]) < d)
    kg = gate(known)
    ccr = np.mean([not g and r['top1'] == r['expected'] for g, r in zip(kg, known)])
    prf = gated_other_prf(kg, gate(unknown)) or {}
    print(f"  rule_a + lề      τ={t:.4f} δ={d:.4f}  biết: nhận+đúng {ccr:.4f}  |  Other: P {prf.get('precision')}  R {prf.get('recall')}  F1 {prf.get('f1')}")
    s = open_set or o['open_set']
    print(f"  AUROC: cos {s['auroc']}   lề {s.get('margin_auroc')}")

# 9a. Chọn τ (quy tắc A: τ@TPR95; quy tắc B: bỏ prototype của chính lớp, có/không loại cặp ★ cos > 0.95), val_sel
import json
for tag, enc, protos in (('lo', '__global_lo__.pt', 'iso_prototypes_lo.pt'), ('full', '__global__.pt', 'iso_prototypes.pt')):
    if not os.path.exists(f'{ENC_DIR}/{enc}'):
        print('bỏ qua (chưa có)', enc); continue
    print(f'===== tau {enc}')
    !cd /content/pid && python src/run_pipeline.py evaluate \
        data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_tau_$tag eval.mode=oneshot ++eval.oneshot.protocol=tau \
        ++eval.image_subset=val_sel oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$protos \
        oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
    o = json.load(open(sorted(glob.glob(f'{RUNS_DIR}/eval_tau_{tag}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
    print(f"{enc}: {o['rule_a']['n_crops']} crop, {o['rule_a']['n_classes']} lớp {o['classes']}")
    print(f"  A  tau={o['rule_a']['tau']:.4f}   B  tau={o['rule_b']['tau']:.4f} (J={o['rule_b']['youden_j']:.3f})   "
          f"B bỏ cặp sinh đôi  tau={o['rule_b_no_twins']['tau']:.4f}")
    m = o['margin_rule_a']
    print(f"  lề top1−top2: δ={m['delta']:.4f} (TPR95)   crop biết qua cả τ_A và δ: {m['known_accepted_with_rule_a_tau']:.3f}")
    print('  cặp ★ cos > 0.95 bị loại:', [(p['class'], p['twin'], round(p['cos'], 3)) for p in o['rule_b_no_twins']['twin_pairs']])

**9b. P1** — encoder leave-out `__global_lo__.pt` + `iso_prototypes_lo.pt`, crop `h_in` / `h_out` từ cả train và val
(encoder chưa train trên chúng; không dùng `image_subset`). Guard rò rỉ: raise nếu lớp thi nằm trong `fine_classes`.
Số chính: AUROC + OSCR (không phụ thuộc τ); top-1 / top-3 macro `h_in`; P/R/F1 của "Other" tại `oneshot.other_min_cosine`.

In [ ]:
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_p1 eval.mode=oneshot ++eval.oneshot.protocol=p1 \
    oneshot.encoder_ckpt=$ENC_DIR/__global_lo__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes_lo.pt \
    oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
M_P1 = sorted(glob.glob(f'{RUNS_DIR}/eval_p1/*/metrics.json'), key=os.path.getmtime)[-1]
o = json.load(open(M_P1))['oneshot']
print(M_P1)
print(f"h_in macro top1 {o['h_in']['macro_top1']:.4f}  top3 {o['h_in']['macro_top3']:.4f}  "
      f"AUROC {o['open_set']['auroc']:.4f}  OSCR {o['open_set']['oscr_area']:.4f}  "
      f"({o['open_set']['n_known']} known / {o['open_set']['n_unknown']} unknown crop)")
print('Tại τ chọn ở 9a (encoder lo):'); at_chosen_taus(o, 'lo', {'H_chon'}, {'OOV_heldout'})
for name, r in o['per_class'].items():
    print(f"  {name:10s} {r['group']:12s} n={r['n']:5d} top1 {r['top1']}  top3 {r['top3']}  hay đoán {r['most_predicted']}")

**9c. P2** — encoder đủ 32 lớp `__global__.pt` + `iso_prototypes.pt`, crop `val` nửa `test`: lớp S (đã thấy) và V
(biến thể) là "biết", `OOV_seen` là "Other đã thấy" (báo riêng, không phải lớp lạ thật: encoder đã train trên chúng).
Gắn nhãn "seen-class" khi báo cáo. Chạy **một lần**, sau khi τ ở 9a đã chốt.

In [ ]:
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_p2 eval.mode=oneshot ++eval.oneshot.protocol=p2 \
    ++eval.image_subset=test oneshot.encoder_ckpt=$ENC_DIR/__global__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes.pt \
    oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
M_P2 = sorted(glob.glob(f'{RUNS_DIR}/eval_p2/*/metrics.json'), key=os.path.getmtime)[-1]
o = json.load(open(M_P2))['oneshot']
print(M_P2)
for g in ('S', 'V', 'V_of_S', 'OOV_seen'):
    b = o['groups'][g]
    print(f"  {g:9s} {b['n_classes']:2d} lớp {b['n_crops']:5d} crop  macro top1 {b['macro_top1']}  top3 {b['macro_top3']}")
print(f"AUROC {o['open_set']['auroc']}  OSCR {o['open_set']['oscr_area']}")
print('Tại τ chọn ở 9a (encoder full):'); at_chosen_taus(o, 'full', {'S', 'V'}, {'OOV_seen'})
print('UNSURE bị loại:', o['unsure_crops_excluded'])

## 10. Baseline chỉ-ImageNet (B4a) — P1 cạnh R1a

Encoder `__imagenet__.pt` = backbone ResNet-34 với trọng số ImageNet mặc định của timm (đúng điểm xuất phát của R1a/R1b),
**không train bước nào** (`scripts/make_imagenet_encoder.py`). `proj` là Identity nên embedding = feature ImageNet thuần
(script dừng nếu không phải). Checkpoint ghi `fine_classes=[]` (head 0 dòng, one-shot không dùng head), `excluded_classes`
= cả 32 lớp, `val_acc=None`: guard rò rỉ P1 qua vì encoder không train lớp nào, không nhờ danh sách gõ tay.

Câu hỏi: train DigitizePID giúp hay **hại** chuyển giao sang lớp chưa thấy? So sánh chính: AUROC + OSCR (không phụ thuộc
τ) và top-1 / top-3 macro `h_in`, trên đúng các crop của 9b. Cần mục 3 (crop), 4, 5b (`__global_lo__.pt`, cho τ ở 10b),
9a (hàm `at_chosen_taus` + τ của R1a) và 9b (P1 của R1a). 10a không cần GPU.

In [ ]:
# 10a. Encoder ImageNet + prototype của nó (cùng template, cùng view như mục 7), rồi separation cạnh R1a / R1b.
import torch
from utils.oneshot import PrototypeDirectory
!cd /content/pid && python scripts/make_imagenet_encoder.py --out $ENC_DIR/__imagenet__.pt \
    data_root=$LOCAL_DATA_ROOT 2>&1 | grep -v "^Warning"
st = torch.load(f'{ENC_DIR}/__imagenet__.pt', map_location='cpu', weights_only=False)
assert st['fine_classes'] == [] and sorted(st['excluded_classes']) == sorted(CM.fine_classes), st['excluded_classes']
print(f"__imagenet__.pt: {st['pretrained_weights']}, head {tuple(st['model']['head.weight'].shape)}, val_acc {st['val_acc']}")
!cd /content/pid && python src/run_pipeline.py prototypes run_dir=$RUNS_DIR/r2 \
    oneshot.encoder_ckpt=$ENC_DIR/__imagenet__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes_imagenet.pt \
    oneshot.symbols_dir=$REPO_DATA/iso_symbols_clean oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv \
    2>&1 | grep -v "^Warning"
for f in ('iso_prototypes_imagenet.pt', 'iso_prototypes_lo.pt', 'iso_prototypes.pt'):
    if not os.path.exists(f'{ENC_DIR}/{f}'):
        print('bỏ qua (chưa có)', f); continue
    d = PrototypeDirectory.load(f'{ENC_DIR}/{f}')
    g = d.embeddings @ d.embeddings.T
    off = g[~torch.eye(len(d), dtype=torch.bool)]
    i, j = divmod(int(torch.argmax(g - 2 * torch.eye(len(d)))), len(d))
    print(f"  {f:28s} mean cos {float(off.mean()):.3f}  max {float(off.max()):.3f} ({d.names[i]} vs {d.names[j]})  "
          f"cặp > 0.95: {int((off > 0.95).sum()) // 2}  > 0.8: {int((off > 0.8).sum()) // 2}")

**10b. τ của encoder ImageNet** — quy tắc A / B như 9a, trên crop `val_sel` của **đúng các lớp R1a dùng ở 9a** (nhóm S
của R1a có prototype: S1, S17, S21, S24). Encoder ImageNet chưa train lớp nào nên tự nó không có nhóm S:
`++eval.oneshot.groups_ckpt=$ENC_DIR/__global_lo__.pt` lấy nhóm lớp từ checkpoint R1a. Code chặn nếu tập đó chứa lớp nào
của `oneshot.holdout`; guard rò rỉ P1 vẫn kiểm encoder thật được nạp. Không ghi τ vào config.

In [ ]:
import json
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_tau_imagenet eval.mode=oneshot ++eval.oneshot.protocol=tau \
    ++eval.image_subset=val_sel ++eval.oneshot.groups_ckpt=$ENC_DIR/__global_lo__.pt \
    oneshot.encoder_ckpt=$ENC_DIR/__imagenet__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes_imagenet.pt \
    oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
o, o_lo = latest('eval_tau_imagenet'), latest('eval_tau_lo')
assert o['classes'] == o_lo['classes'], (o['classes'], o_lo['classes'])  # cùng lớp với τ của R1a (9a)
print(f"__imagenet__.pt: {o['rule_a']['n_crops']} crop, {o['rule_a']['n_classes']} lớp {o['classes']}")
for name, t in (('ImageNet', o), ('R1a lo', o_lo)):
    print(f"  {name:9s} A tau={t['rule_a']['tau']:.4f}   B tau={t['rule_b']['tau']:.4f} (J={t['rule_b']['youden_j']:.3f})   "
          f"B bỏ cặp sinh đôi tau={t['rule_b_no_twins']['tau']:.4f}   δ lề={t.get('margin_rule_a', {}).get('delta')}")
print('  cặp ★ cos > 0.95 bị loại (ImageNet):', [(p['class'], p['twin'], round(p['cos'], 3)) for p in o['rule_b_no_twins']['twin_pairs']])

**10c. P1 với encoder ImageNet** — đúng lệnh 9b, chỉ đổi encoder + prototype (crop `h_in` / `h_out` từ train + val, không
`image_subset`), rồi in cạnh R1a (lượt 9b mới nhất trong `eval_p1/`). Mức ngẫu nhiên: top-1 = 1/42 ≈ 0.024, top-3 ≈ 0.071.
ImageNet ≥ R1a trên AUROC / OSCR / top-1 → train DigitizePID không giúp (hoặc hại) chuyển giao sang lớp chưa thấy.

In [ ]:
import json
!cd /content/pid && python src/run_pipeline.py evaluate \
    data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_p1_imagenet eval.mode=oneshot ++eval.oneshot.protocol=p1 \
    oneshot.encoder_ckpt=$ENC_DIR/__imagenet__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes_imagenet.pt \
    oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean 2>&1 | grep -v "^Warning"
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
im, lo = latest('eval_p1_imagenet'), latest('eval_p1')
assert im['n_crops'] == lo['n_crops'], (im['n_crops'], lo['n_crops'])  # cùng crop với 9b
fmt = lambda v: '     -' if v is None else f'{v:6.4f}'
print(f"{'':22s} {'ImageNet':>8s} {'R1a lo':>8s}   (ngẫu nhiên: top1 {1 / im['n_prototypes']:.3f}, top3 {3 / im['n_prototypes']:.3f})")
for label, key in (('AUROC', lambda o: o['open_set']['auroc']), ('OSCR', lambda o: o['open_set']['oscr_area']),
                   ('h_in macro top1', lambda o: o['h_in']['macro_top1']), ('h_in macro top3', lambda o: o['h_in']['macro_top3'])):
    print(f"  {label:20s} {fmt(key(im)):>8s} {fmt(key(lo)):>8s}")
print(f"  ({im['open_set']['n_known']} known / {im['open_set']['n_unknown']} unknown crop)")
if 'at_chosen_taus' in globals():
    print('Tại τ chọn ở 10b (ImageNet):'); at_chosen_taus(im, 'imagenet', {'H_chon'}, {'OOV_heldout'})
    print('Tại τ chọn ở 9a (R1a lo):'); at_chosen_taus(lo, 'lo', {'H_chon'}, {'OOV_heldout'})
else:
    print('(chạy phần def đầu cell 9a để có at_chosen_taus — không cần chạy lại τ)')
print('Từng lớp, ImageNet | R1a lo:')
for name, a in im['per_class'].items():
    b = lo['per_class'][name]
    print(f"  {name:10s} {a['group']:12s} n={a['n']:5d}  top1 {fmt(a['top1'])} | {fmt(b['top1'])}  top3 {fmt(a['top3'])} | {fmt(b['top3'])}  "
          f"cos {fmt(a['mean_best_cos'])} | {fmt(b['mean_best_cos'])}  hay đoán {a['most_predicted']} | {b['most_predicted']}")

**10d. CL2N — căn giữa trước khi so cosine** (SimpleShot, Wang et al. 2019, arXiv 1911.04623). Feature ImageNet sau
average-pool đều ≥ 0 nên mọi cos đều cao và prototype gần hướng trung bình của crop thắng gần hết (10c: 59% crop P1 →
`flow_direction`). CL2N trừ vector trung bình rồi mới chuẩn hoá L2, cho **cả** crop lẫn prototype. Tâm = trung bình
embedding mọi view của crop **train** các lớp R1a đã thấy, trừ mọi lớp `oneshot.holdout` (`++eval.oneshot.protocol=center`,
nhóm lớp lấy từ `__global_lo__.pt` như 10b) → không mang nhãn lớp nào P1 thi. Mỗi encoder một tâm riêng; τ phải chọn lại
(cos đổi không gian) — vẫn trên `val_sel`, cùng quy tắc 9a. Đo nhanh cục bộ 2026-11-15 (mẫu 1,500 crop P1): ImageNet top-1
0.010 → 0.133, top-3 0.12 → 0.41, AUROC 0.48 → 0.43; R1a gần như không đổi (top-1 0.044 → 0.041, AUROC 0.78 → 0.81).
Tâm bằng trung bình 42 prototype thì **tệ hơn** (miền template ≠ miền crop) — không dùng.

In [ ]:
# 10d. Tâm CL2N cho ImageNet và R1a, rồi τ (val_sel) + P1 lại với ++oneshot.center. Cần mục 3 (crop train), 10a–10c, 9b.
import json, shutil
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
GROUPS = f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global_lo__.pt'
for tag, enc, protos in (('imagenet', '__imagenet__.pt', 'iso_prototypes_imagenet.pt'),
                         ('lo', '__global_lo__.pt', 'iso_prototypes_lo.pt')):
    if not os.path.exists(f'{ENC_DIR}/{enc}'):
        print('bỏ qua (chưa có)', enc); continue
    BASE = (f'data_root={LOCAL_DATA_ROOT} oneshot.encoder_ckpt={ENC_DIR}/{enc} oneshot.prototypes={ENC_DIR}/{protos} '
            f'oneshot.vocab_csv={REPO_DATA}/iso_symbols/vocab.csv oneshot.aggregate=mean')
    CENTER = f'{ENC_DIR}/center_{tag}.pt'
    RUN_C, RUN_T, RUN_P = (f'{RUNS_DIR}/eval_{p}_{tag}' + ('' if p == 'center' else '_cl2n') for p in ('center', 'tau', 'p1'))
    print(f'===== tâm {enc}')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUN_C eval.mode=oneshot ++eval.oneshot.protocol=center \
        $GROUPS ++eval.oneshot.max_per_class=200 $BASE 2>&1 | grep -v "^Warning"
    c = latest(f'eval_center_{tag}')
    shutil.copy2(c['center_file'], CENTER)
    print(f"  {c['n_crops']} crop ({c['n_views']} view), {len(c['classes'])} lớp, bỏ lớp giữ ra {c['held_out_dropped']}, "
          f"|tâm|={c['center_norm']:.3f} -> {CENTER}")
    print(f'===== tau {enc} + CL2N')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUN_T eval.mode=oneshot ++eval.oneshot.protocol=tau \
        ++eval.image_subset=val_sel $GROUPS ++oneshot.center=$CENTER $BASE 2>&1 | grep -v "^Warning"
    o = latest(f'eval_tau_{tag}_cl2n')
    assert o['classes'] == latest('eval_tau_lo')['classes'], o['classes']  # cùng lớp với τ của 9a
    print(f"  A tau={o['rule_a']['tau']:.4f}   B tau={o['rule_b']['tau']:.4f} (J={o['rule_b']['youden_j']:.3f})   "
          f"δ lề={o['margin_rule_a']['delta']:.4f}")
    print(f'===== P1 {enc} + CL2N')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUN_P eval.mode=oneshot ++eval.oneshot.protocol=p1 \
        ++oneshot.center=$CENTER $BASE 2>&1 | grep -v "^Warning"

**10e. So bốn encoder trên P1** (cùng crop với 9b / 10c): ImageNet, ImageNet + CL2N, R1a, R1a + CL2N. "hub" = tỉ lệ crop
(biết + lạ) bị gán cho prototype hay ra nhất — ImageNet thô ~0.59 là dấu hiệu hub. Kết luận "train DigitizePID hại chuyển
giao" chỉ vững nếu so với ImageNet **đã căn giữa**.

In [ ]:
# 10e. Bảng P1: ImageNet / ImageNet+CL2N / R1a / R1a+CL2N, rồi τ đã chọn lại (10d) cho bản CL2N.
import csv as _csv
from collections import Counter
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
fmt = lambda v: '     -' if v is None else f'{v:6.4f}'
RUNS = (('ImageNet', 'eval_p1_imagenet', 'imagenet'), ('ImageNet+CL2N', 'eval_p1_imagenet_cl2n', 'imagenet_cl2n'),
        ('R1a', 'eval_p1', 'lo'), ('R1a+CL2N', 'eval_p1_lo_cl2n', 'lo_cl2n'))
have = [(n, latest(r), t) for n, r, t in RUNS if glob.glob(f'{RUNS_DIR}/{r}/*/metrics.json')]
assert len({o['n_crops'] for _, o, _ in have}) == 1, [(n, o['n_crops']) for n, o, _ in have]  # cùng crop
for n, o, _ in have:
    assert ('CL2N' in n) == bool(o.get('center')), (n, o.get('center'))
def hub(o):
    top = Counter(r['top1'] for r in _csv.DictReader(open(o['crops_csv'])))
    name, k = top.most_common(1)[0]
    return name, k / sum(top.values())
print(f"{'':18s}" + ''.join(f'{n:>15s}' for n, _, _ in have))
for label, key in (('AUROC (cos)', lambda o: o['open_set']['auroc']), ('AUROC (lề)', lambda o: o['open_set'].get('margin_auroc')),
                   ('OSCR', lambda o: o['open_set']['oscr_area']), ('h_in macro top1', lambda o: o['h_in']['macro_top1']),
                   ('h_in macro top3', lambda o: o['h_in']['macro_top3']), ('hub (tỉ lệ)', lambda o: hub(o)[1])):
    print(f'  {label:16s}' + ''.join(f'{fmt(key(o)):>15s}' for _, o, _ in have))
print('  hub (tên)       ' + ''.join(f'{hub(o)[0][:14]:>15s}' for _, o, _ in have))
print(f"  ({have[0][1]['open_set']['n_known']} known / {have[0][1]['open_set']['n_unknown']} unknown crop; "
      f"ngẫu nhiên top1 {1 / have[0][1]['n_prototypes']:.3f}, top3 {3 / have[0][1]['n_prototypes']:.3f})")
if 'at_chosen_taus' in globals():
    for n, o, t in have:
        if 'CL2N' in n:
            print(f'Tại τ chọn ở 10d ({n}):'); at_chosen_taus(o, t, {'H_chon'}, {'OOV_heldout'})
else:
    print('(chạy phần def đầu cell 9a để có at_chosen_taus — không cần chạy lại τ)')
print('Từng lớp top1: ' + ' | '.join(n for n, _, _ in have))
for name in have[0][1]['per_class']:
    a = [o['per_class'][name] for _, o, _ in have]
    print(f"  {name:10s} {a[0]['group']:12s}  top1 " + ' | '.join(fmt(x['top1']) for x in a)
          + '   hay đoán ' + ' | '.join(str(x['most_predicted']) for x in a))

## 11. P4 — OPEN100-ISO mức crop (đề thi "chuyển sang ISO")

Mỗi box GT của export V8 (`iso00`…`iso11`, 36 biến thể/tờ, bản sạch + bản suy giảm) được cắt từ bản vẽ với lề 10% như lúc
suy luận (`utils.crop_box.cut`), nhúng như §9 (`oneshot.aggregate=mean`), so với **mọi** prototype. Nhóm của mỗi lớp ISO tính
tự động từ `symbol_map.csv` + `fine_classes` / `excluded_classes` của checkpoint (PLAN "Lớp ISO thuộc nhóm nào"):
**S / V / H_chon / H_tu_nhien** (+ `V_held`: chỉ có lớp VARIANT bị giữ ra map tới — chỉ gặp ở encoder không train lớp nào).
Mỗi nhóm một dòng, không gộp. "Other": dương = `other_iso`, âm = mọi instance vocab; `other_isa` (bong bóng ISA, "Other đã
thấy" với R1b) và `other_generic` báo riêng.

Quy tắc báo cáo (PLAN B6b, 2026-11-11): **không bao giờ gộp sạch và suy giảm** — số chính = ảnh suy giảm, ảnh sạch là
tham chiếu; mỗi tờ gốc một số rồi trung bình ± độ lệch trên 12 tờ; kèm số instance. **Không chọn gì trên OPEN100-ISO**:
τ / δ lấy từ §9a (R1a, R1b) và §10b (ImageNet). Encoder ImageNet chấm theo nhóm của R1a
(`++eval.oneshot.groups_ckpt=$ENC_DIR/__global_lo__.pt`, như 10b) để đặt cạnh R1a đúng cùng tập lớp. Mỗi lượt ghi
`runs/eval_p4_<tag>/<ts>/metrics.json` + `oneshot_p4_crops.csv` (thêm cột `sheet`, `variant`, `degraded`, `ann_id`: mọi số
theo ngưỡng tính lại được không cần GPU). Cần mục 7, 9a (`at_chosen_taus` + τ), 10a/10b (encoder ImageNet + τ của nó).

**Dữ liệu (chưa có trên Colab)**: ở máy local, `cd data/iso_open100 && zip -r -0 iso_open100_canonical.zip canonical`
(~1.9 GB: 864 ảnh + `annotations.coco.json`, `classes.yaml`, `manifest.csv`; `-0` vì PNG/JPG đã nén sẵn), rồi upload
thành `<DRIVE_ROOT>/data/iso_open100_canonical.zip`. Cell 11a giải nén vào `/content/iso_open100`.

In [ ]:
# ▶▶ 11a. OPEN100-ISO: giải nén từ Drive (CHƯA có trên Colab — tạo zip ở local, xem markdown trên) ◀◀
import json, time
ISO_ZIP = f'{DATA_ROOT}/iso_open100_canonical.zip'
LOCAL_ISO_ROOT = '/content/iso_open100'
if not os.path.exists(f'{LOCAL_ISO_ROOT}/canonical/annotations.coco.json'):
    assert os.path.exists(ISO_ZIP), f'Chưa có {ISO_ZIP}: ở local chạy `cd data/iso_open100 && zip -r -0 iso_open100_canonical.zip canonical` rồi upload'
    t0 = time.time()
    !mkdir -p "$LOCAL_ISO_ROOT" && unzip -q -o "$ISO_ZIP" -d "$LOCAL_ISO_ROOT/"
    print(f'Giải nén sau {time.time()-t0:.0f}s')
coco = json.load(open(f'{LOCAL_ISO_ROOT}/canonical/annotations.coco.json'))
n_img = len(os.listdir(f'{LOCAL_ISO_ROOT}/canonical/images'))
n_sym = sum(not a.get('iscrowd') for a in coco['annotations'])   # vùng bỏ qua (iscrowd, B6d.2) không phải ký hiệu
print(f"{n_img} ảnh, {len(coco['images'])} ảnh COCO ({sum(im['degraded'] for im in coco['images'])} suy giảm), "
      f"{n_sym} annotation ký hiệu + {len(coco['annotations']) - n_sym} vùng bỏ qua, {len(coco['categories'])} lớp")
assert n_img == len(coco['images']) == 864 and n_sym == 93800, 'zip OPEN100-ISO không phải export V8?'
del coco

**11b. P4 cho ba encoder** — R1b (`__global__.pt` + `iso_prototypes.pt`), R1a (`__global_lo__.pt` + `iso_prototypes_lo.pt`),
ImageNet (`__imagenet__.pt` + `iso_prototypes_imagenet.pt`, nhóm theo R1a). Không `image_subset` (OPEN100-ISO không có nửa
`val_sel` / `test`: code chặn). Đủ ~47k crop mỗi phiên bản ảnh (sạch / suy giảm) mỗi encoder; chạy thử nhanh:
`++eval.oneshot.max_per_class=30` (lấy mẫu theo (lớp, sạch/suy giảm) với seed cố định, ghi vào metrics.json).

In [ ]:
P4_RUNS = (('full', '__global__.pt', 'iso_prototypes.pt', ''),
           ('lo', '__global_lo__.pt', 'iso_prototypes_lo.pt', ''),
           ('imagenet', '__imagenet__.pt', 'iso_prototypes_imagenet.pt', f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global_lo__.pt'))
for tag, enc, protos, extra in P4_RUNS:
    if not (os.path.exists(f'{ENC_DIR}/{enc}') and os.path.exists(f'{ENC_DIR}/{protos}')):
        print('bỏ qua (chưa có)', enc, protos); continue
    print(f'===== P4 {enc}')
    !cd /content/pid && python src/run_pipeline.py evaluate \
        data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/eval_p4_$tag eval.mode=oneshot ++eval.oneshot.protocol=p4 \
        oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$protos \
        oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean $extra 2>&1 | grep -v "^Warning"

**11c. Bảng cạnh nhau** — suy giảm (số chính) rồi sạch (tham chiếu). Mỗi ô nhóm: top-1 / top-3 macro theo lớp, top-1 theo
tờ (trung bình ± độ lệch trên 12 tờ), (số lớp / số crop). "Other": AUROC trên cos và trên lề top1−top2, OSCR, AUROC theo tờ;
`other_isa` / `other_generic`: AUROC riêng so với vocab. Sau đó P/R/F1 của "Other" (`other_iso`) tại τ / δ của 9a / 10b
(`at_chosen_taus`, lọc theo cột `split` của CSV) và tỉ lệ `other_isa` / `other_generic` bị loại tại τ_A của encoder đó.

In [ ]:
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
P4 = {tag: latest(f'eval_p4_{tag}') for tag, *_ in P4_RUNS if glob.glob(f'{RUNS_DIR}/eval_p4_{tag}/*/metrics.json')}
P4_NAME = {'full': 'R1b', 'lo': 'R1a', 'imagenet': 'ImageNet'}
P4_GROUPS = ('S', 'V', 'H_chon', 'V_held', 'H_tu_nhien')
fmt = lambda v: '    -' if v is None else f'{v:5.3f}'
pm = lambda ms: '     -     ' if ms['mean'] is None else f"{ms['mean']:.3f}±{ms['std'] or 0:.3f}"
print('mẫu:', {P4_NAME[t]: o['max_per_class'] for t, o in P4.items()}, ' τ trong metrics.json:', {P4_NAME[t]: o['other_min_cosine'] for t, o in P4.items()})
for view in ('degraded', 'clean'):
    print(f"\n===== {view} ({'SỐ CHÍNH' if view == 'degraded' else 'tham chiếu'})   ô: top1 top3 | top1 theo tờ | (lớp/crop)")
    print(f"  {'':11s}" + ''.join(f'{P4_NAME[t]:>40s}' for t in P4))
    for g in P4_GROUPS:
        cells = []
        for o in P4.values():
            b, ms = o[view]['groups'][g], o[view]['sheet_mean_std']['groups'][g]['macro_top1']
            cells.append(f"{fmt(b['macro_top1'])} {fmt(b['macro_top3'])} | {pm(ms)} ({b['n_classes']:2d}/{b['n_crops']:5d})")
        print(f'  {g:11s}' + ''.join(f'{c:>40s}' for c in cells))
    for label, key in (('AUROC cos', lambda b: fmt(b['open_set']['auroc'])), ('AUROC lề', lambda b: fmt(b['open_set']['margin_auroc'])),
                       ('OSCR', lambda b: fmt(b['open_set']['oscr_area'])), ('AUROC/tờ', lambda b: pm(b['sheet_mean_std']['auroc'])),
                       ('isa AUROC', lambda b: fmt(b['other']['other_isa']['auroc'])),
                       ('generic AUROC', lambda b: fmt(b['other']['other_generic']['auroc']))):
        print(f'  {label:13s}' + ''.join(f'{key(o[view]):>38s}' for o in P4.values()))
    print('  n: ' + '  '.join(f"{P4_NAME[t]} {o[view]['open_set']['n_known']} vocab / {o[view]['open_set']['n_unknown']} other_iso / "
                              f"{o[view]['other']['other_isa']['n']} isa / {o[view]['other']['other_generic']['n']} generic" for t, o in P4.items()))
if 'at_chosen_taus' not in globals():
    print('\n(chạy phần def đầu cell 9a để có at_chosen_taus — không cần chạy lại τ)')
else:
    for tag, o in P4.items():
        rows = list(_csv.DictReader(open(o['crops_csv'])))
        t = chosen_taus(tag)['rule_a']
        for view in ('degraded', 'clean'):
            print(f'\n{P4_NAME[tag]} — {view}, tại τ / δ chọn ở 9a / 10b:')
            at_chosen_taus(o, tag, set(P4_GROUPS), {'other_iso'}, keep=lambda r: r['split'] == view, open_set=o[view]['open_set'])
            for name in ('other_isa', 'other_generic'):
                cos = [float(r['best_cos']) for r in rows if r['split'] == view and r['group'] == name]
                print(f"  {name:13s} bị loại tại τ_A={t:.4f}: {np.mean([c < t for c in cos]) if cos else None}  (n={len(cos)})")

## 12. Stage 1 + end-to-end trên OPEN100-ISO (B6b-2) + box sai của detector

Ba phần, đều chấm bằng `evaluate` (CPU, rẻ) trên file detection đã ghi; phần tốn GPU là detector (12a) và nhúng crop (12b, 12c).

- **12a. Stage 1 không phân lớp** — detector coarse 8 lớp DigitizePID chạy trên OPEN100-ISO (`data_root=$LOCAL_ISO_ROOT`) hai
  lượt: NMS/WBF theo nhãn (mặc định) và `++detector.agnostic_merge=true` (B6e). `++detector.classes_yaml` = `classes.yaml`
  DigitizePID (8 lớp coarse của checkpoint; `classes.yaml` của OPEN100-ISO là lớp ISO). Chấm `eval.mode=det_only
  ++eval.class_agnostic=true`: mọi box là một lớp "symbol"; P / R (IoU 0.5, `eval.score_thr`), mAP@50, mAP@50:95; recall
  **theo nhóm** S / V / H_chon / H_tu_nhien / other_* (nhóm tính từ checkpoint encoder như P4 — chấm hai lần, nhóm theo R1b
  và theo R1a) và theo lớp. **H_tu_nhien** = "detector DigitizePID có thấy hình chưa từng thấy không" — trần của end-to-end.
- **12b. End-to-end P4** — Stage 1 (lượt `agnostic_merge`) + one-shot Stage 2 (`fgc --infer ++fgc.oneshot=true`, ghi
  `derived/detections_oneshot/`, cos **chưa gate**) cho R1b và R1a; chấm `eval.mode=det_plus_fgc ++fgc.oneshot=true`: TP khi
  IoU ≥ 0.5 **và** đúng nhãn ISO; "Other" (cos < τ, hoặc thêm lề < δ) đúng với GT `other_iso` / `other_isa` / `other_generic`
  (báo riêng từng dòng). τ / δ lấy từ **9a** (cổng đặt lúc chấm, đổi τ không phải nhúng lại); không chọn gì trên OPEN100-ISO.
- **12c. Box sai của detector** (PLAN B6b) — DigitizePID `val`, chấm nửa `val_sel`: box dự đoán có IoU < 0.3 với mọi GT → tỉ lệ
  bị τ loại (và τ HOẶC lề < δ). Cùng lệnh in thêm tỉ lệ loại nhầm trên box đúng (tham chiếu).

Quy tắc báo cáo (PLAN 2026-11-11): code tự tách **suy giảm (số chính) / sạch (tham chiếu)**, mỗi tờ gốc một số rồi trung bình ±
độ lệch trên 12 tờ, kèm số instance. Vùng bỏ qua (mũi tên dẫn, cờ: `iscrowd`, sau lượt export lại B6d.2) không vào mẫu số recall;
dự đoán rơi vào đó (IoU ≥ 0.5 hoặc ≥ 50% diện tích dự đoán nằm trong vùng) không TP cũng không FP, được đếm theo loại.

**Cần**: mục 3 (copy `classes.yaml` DigitizePID vào `$LOCAL_DATA_ROOT`), 7 (prototype), **9a** (τ / δ + hàm `chosen_taus`), 11a
(OPEN100-ISO). **`DET_CKPT`** (cell 12a): `best.pt` của run detector coarse (notebook Stage 1 / RUNBOOK §4, §6) — repo không ghi
đường dẫn này, **phải sửa tay**. Detection lưu lên Drive (`runs/stage1_iso/`, `runs/stage1_dpid/`): phiên sau dùng lại, không
chạy lại detector. Thời gian (T4, ước lượng): detector ~60 tile/tờ × 864 tờ mỗi lượt; Stage 2 ~110 crop/tờ × 864 tờ mỗi encoder.

In [ ]:
# ▶▶ 12a. Stage 1 trên OPEN100-ISO, hai cách gộp box, chấm không phân lớp — SỬA DET_CKPT ◀◀
import json, shutil, time
DET_CKPT = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX/train/weights/best.pt'   # <-- sửa: best.pt của detector coarse 8 lớp (Stage 1)
if not os.path.exists(DET_CKPT):
    print('Ứng viên trên Drive:', sorted(glob.glob(f'{RUNS_DIR}/*/train/weights/best.pt')))
assert os.path.exists(DET_CKPT), f'Không thấy {DET_CKPT}: sửa DET_CKPT (ứng viên in ở trên)'
DET_CLASSES = f'{LOCAL_DATA_ROOT}/canonical/classes.yaml'   # 8 lớp coarse DigitizePID (mục 3 đã copy)
assert len(load_classmap(DET_CLASSES).coarse_classes) == 8, DET_CLASSES
DET = f'detector.ckpt={DET_CKPT} detector.label_level=coarse'
ISO_DET = f'{LOCAL_ISO_ROOT}/derived/detections/test'
KEEP_ISO = f'{RUNS_DIR}/stage1_iso'          # detection lưu trên Drive, phiên sau dùng lại
MERGES = (('perlabel', ''), ('agnostic', '++detector.agnostic_merge=true'))
GROUPS_BY = (('full', '__global__.pt'), ('lo', '__global_lo__.pt'))   # nhóm S / V / H theo R1b, theo R1a
E2E_RUNS = (('full', '__global__.pt', 'iso_prototypes.pt'), ('lo', '__global_lo__.pt', 'iso_prototypes_lo.pt'))   # 12b, 12c
OS_ARGS = f'oneshot.vocab_csv={REPO_DATA}/iso_symbols/vocab.csv oneshot.aggregate=mean'   # như §9 / §11
NAME = {'full': 'R1b', 'lo': 'R1a', 'perlabel': 'gộp theo nhãn', 'agnostic': 'agnostic_merge'}
metrics_of = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))
fmt = lambda v: '    -' if v is None else f'{v:5.3f}'
pm = lambda ms: '-' if ms is None or ms['mean'] is None else f"{ms['mean']:.3f}±{ms['std'] or 0:.3f}"
def restore(keep, dst, n):
    # chép một bộ detection đã lưu vào chỗ evaluate / fgc --infer đọc
    assert len(glob.glob(f'{keep}/*.txt')) == n, (keep, len(glob.glob(f'{keep}/*.txt')))
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree(keep, dst)
for tag, merge in MERGES:
    keep = f'{KEEP_ISO}/detections_{tag}'
    if len(glob.glob(f'{keep}/*.txt')) != 864:
        t0 = time.time()
        shutil.rmtree(ISO_DET, ignore_errors=True)
        !cd /content/pid && python src/run_pipeline.py detect --infer \
            data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/det_iso_$tag eval.split=test $DET \
            ++detector.classes_yaml=$DET_CLASSES $merge 2>&1 | grep -v "^Warning" | tail -2
        restore(ISO_DET, keep, 864)
        print(f'detector {tag}: {time.time() - t0:.0f}s')
    restore(keep, ISO_DET, 864)
    for g, enc in GROUPS_BY:
        print(f'===== Stage 1 {tag}, nhóm theo {enc}')
        !cd /content/pid && python src/run_pipeline.py evaluate \
            data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/eval_det_iso_{tag}_{g} eval.mode=det_only eval.split=test \
            detector.label_level=coarse ++eval.class_agnostic=true oneshot.encoder_ckpt=$ENC_DIR/$enc \
            oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv 2>&1 | grep -v "^Warning" | grep "detection"

In [ ]:
# 12a (bảng). Không phân lớp: P / R / mAP, dự đoán trong vùng bỏ qua, recall theo nhóm (± độ lệch trên 12 tờ)
D12 = {(t, g): metrics_of(f'eval_det_iso_{t}_{g}')['detection'] for t, _ in MERGES for g, _ in GROUPS_BY
       if glob.glob(f'{RUNS_DIR}/eval_det_iso_{t}_{g}/*/metrics.json')}
for view in ('degraded', 'clean'):
    print(f"\n===== {view} ({'SỐ CHÍNH' if view == 'degraded' else 'tham chiếu'})")
    for t, _ in MERGES:
        if (t, 'full') not in D12:
            continue
        b = D12[(t, 'full')][view]; ms = b['sheet_mean_std']; ig = b.get('ignore')
        print(f"  {NAME[t]:15s} P {fmt(b['precision'])}  R {fmt(b['recall'])} (theo tờ {pm(ms['recall'])})  mAP50 {fmt(b['map_50'])}"
              f"  mAP50:95 {fmt(b['map_50_95'])}  ({b['n_gt']} GT, {b['n_predictions']} dự đoán, TP {b['tp']} FP {b['fp']})")
        print(f"  {'':15s} trong vùng bỏ qua: " + (f"{ig['predictions_by_kind']} / {ig['regions_by_kind']} vùng" if ig else 'export chưa có vùng bỏ qua'))
        for g, _ in GROUPS_BY:
            if (t, g) not in D12:
                continue
            bg = D12[(t, g)][view]; msg = bg['sheet_mean_std'].get('recall_by_group', {})
            print(f"  {'':15s} recall nhóm ({NAME[g]}): " + '  '.join(
                f"{k} {fmt(r['recall'])} ({r['tp']}/{r['n']}; tờ {pm(msg.get(k, {}).get('recall'))})" for k, r in bg['recall_by_group'].items()))
if ('agnostic', 'full') in D12:
    print('\n10 lớp recall thấp nhất (agnostic_merge, suy giảm):')
    rows = D12[('agnostic', 'full')]['degraded']['recall_by_class']
    for name, r in sorted(rows.items(), key=lambda kv: kv[1]['recall'])[:10]:
        print(f"  {name:32s} {r['group']:11s} recall {fmt(r['recall'])} ({r['tp']}/{r['n']})")

**12b. End-to-end P4** — Stage 1 = lượt `agnostic_merge` của 12a (một ký hiệu lạ = một box, một crop). Mỗi encoder: `fgc --infer
++fgc.oneshot=true` (crop + 10%, `oneshot.aggregate=mean`, so với **mọi** prototype, không route theo coarse) rồi `evaluate` hai lần:
tại τ quy tắc A của 9a, và τ_A + δ lề (nếu 9a có δ). Encoder ImageNet không chạy (không có Stage 2 riêng để so ở đây).

In [ ]:
# 12b. End-to-end: Stage 1 (agnostic_merge) + one-shot Stage 2, R1b và R1a, tại τ / δ của 9a
assert 'chosen_taus' in globals(), 'chạy phần def đầu cell 9a (at_chosen_taus / chosen_taus) — không cần chạy lại τ'
ISO_OS = f'{LOCAL_ISO_ROOT}/derived/detections_oneshot/test'
restore(f'{KEEP_ISO}/detections_agnostic', ISO_DET, 864)
for tag, enc, protos in E2E_RUNS:
    if not (os.path.exists(f'{ENC_DIR}/{enc}') and os.path.exists(f'{ENC_DIR}/{protos}')):
        print('bỏ qua (chưa có)', enc, protos); continue
    tau, delta = chosen_taus(tag)['rule_a'], chosen_delta(tag)
    keep = f'{KEEP_ISO}/detections_oneshot_{tag}'
    if len(glob.glob(f'{keep}/*.txt')) != 864:
        t0 = time.time()
        shutil.rmtree(ISO_OS, ignore_errors=True)
        !cd /content/pid && python src/run_pipeline.py fgc --infer \
            data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/e2e_iso_$tag eval.split=test ++fgc.oneshot=true \
            ++detector.classes_yaml=$DET_CLASSES oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$protos \
            oneshot.other_min_cosine=$tau $OS_ARGS 2>&1 | grep -v "^Warning" | tail -2
        restore(ISO_OS, keep, 864)
        print(f'Stage 2 {enc}: {time.time() - t0:.0f}s')
    restore(keep, ISO_OS, 864)
    for suffix, margin in (('', ''), ('_margin', '' if delta is None else f'++eval.oneshot.margin_delta={delta}')):
        if suffix and not margin:
            print('  (9a chưa có δ: bỏ lượt τ + lề)'); continue
        print(f'===== e2e {enc}  τ={tau:.4f}' + (f'  δ={delta:.4f}' if margin else ''))
        !cd /content/pid && python src/run_pipeline.py evaluate \
            data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/eval_e2e_iso_{tag}{suffix} eval.mode=det_plus_fgc eval.split=test \
            ++fgc.oneshot=true oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.other_min_cosine=$tau $margin $OS_ARGS \
            2>&1 | grep -v "^Warning" | grep "oneshot_e2e"

In [ ]:
# 12b (bảng). Mỗi ô: recall (đúng box + đúng nhãn) | recall box | (GT); "Other" đúng với other_*. Độ chính xác theo nhãn dự đoán.
E2E = {(t, s): metrics_of(f'eval_e2e_iso_{t}{s}')['oneshot_e2e'] for t, *_ in E2E_RUNS for s in ('', '_margin')
       if glob.glob(f'{RUNS_DIR}/eval_e2e_iso_{t}{s}/*/metrics.json')}
for view in ('degraded', 'clean'):
    print(f"\n===== {view} ({'SỐ CHÍNH' if view == 'degraded' else 'tham chiếu'})")
    for (t, s), o in E2E.items():
        b = o[view]; ms = b['sheet_mean_std']
        gate = f"τ={o['tau']:.4f}" + ('' if o['margin_delta'] is None else f" δ={o['margin_delta']:.4f}")
        print(f"  {NAME[t]} {gate}: P {fmt(b['precision'])} (theo tờ {pm(ms['precision'])})  R {fmt(b['recall'])} (theo tờ {pm(ms['recall'])})"
              f"  R box {fmt(b['box_recall'])}  FP: box sai {b['fp_box']}, nhãn sai {b['fp_label']}  bỏ qua {b['n_ignored_predictions']}")
        for k, r in b['by_group'].items():
            bm = r['box_matched']
            print(f"      {k:11s} {fmt(r['recall'])} | {fmt(r['box_recall'])} ({r['n']:5d})  tờ {pm(ms['by_group'][k]['recall'])}"
                  f"   box khớp: đúng {bm['right']}, Other {bm['rejected']}, sai tên {bm['wrong_name']}")
        print('      độ chính xác theo nhãn dự đoán: ' + '  '.join(
            f"{k} {fmt(r['precision'])} (n={r['n']})" for k, r in b['precision_by_predicted_group'].items()))
        w = b['wrong_boxes']
        print(f"      box sai (IoU<0.3 mọi GT): {w['n']} ({fmt(w['frac_of_predictions'])} số dự đoán), bị loại τ {fmt(w['rejected_tau'])}"
              + ('' if 'rejected_tau_or_margin' not in w else f", τ|lề {fmt(w['rejected_tau_or_margin'])}")
              + f";  box đúng bị loại nhầm τ {fmt(b['true_vocab_boxes']['rejected_tau'])}")

**12c. Box sai của detector trên DigitizePID** (PLAN B6b) — detector coarse trên cả `val` (100 ảnh, ảnh gốc 7168 px đọc từ Drive
qua symlink, không ghi gì vào `$DATA_ROOT`), Stage 2 one-shot cho R1b và R1a, chấm **chỉ nửa `val_sel`**
(`++eval.image_subset=val_sel`, nửa đã dùng chọn τ — không đụng `test`). Gộp box mặc định (theo nhãn, như Stage 1 trên DigitizePID).
Số chính: tỉ lệ box sai bị τ loại (và τ HOẶC lề < δ); kèm tỉ lệ box đúng bị loại nhầm và P / R end-to-end trên `val_sel`.
Cần cell 12a đã chạy trong phiên (`DET_CKPT`, `restore`; detection OPEN100-ISO đã lưu thì 12a không chạy lại detector) và
cell def của 9a (`chosen_taus`); không cần 12b.

In [ ]:
# 12c. Box sai (IoU < 0.3 với mọi GT) trên DigitizePID val_sel: tỉ lệ bị τ / τ+lề loại
assert 'chosen_taus' in globals() and 'restore' in globals(), 'chạy cell def của 9a và cell 12a trước'
if not os.path.exists(f'{LOCAL_DATA_ROOT}/canonical/images'):
    os.symlink(f'{DATA_ROOT}/canonical/images', f'{LOCAL_DATA_ROOT}/canonical/images')   # chỉ đọc
VAL_DET, VAL_OS = f'{LOCAL_DATA_ROOT}/derived/detections/val', f'{LOCAL_DATA_ROOT}/derived/detections_oneshot/val'
KEEP_DPID = f'{RUNS_DIR}/stage1_dpid'
import pandas as pd
N_VAL = int((pd.read_csv(f'{LOCAL_DATA_ROOT}/canonical/manifest.csv')['split'] == 'val').sum())   # 100
if len(glob.glob(f'{KEEP_DPID}/detections_val/*.txt')) != N_VAL:
    shutil.rmtree(VAL_DET, ignore_errors=True)
    !cd /content/pid && python src/run_pipeline.py detect --infer \
        data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/det_dpid_val eval.split=val $DET 2>&1 | grep -v "^Warning" | tail -2
    restore(VAL_DET, f'{KEEP_DPID}/detections_val', N_VAL)
restore(f'{KEEP_DPID}/detections_val', VAL_DET, N_VAL)
for tag, enc, protos in E2E_RUNS:
    if not (os.path.exists(f'{ENC_DIR}/{enc}') and os.path.exists(f'{ENC_DIR}/{protos}')):
        print('bỏ qua (chưa có)', enc, protos); continue
    tau, delta = chosen_taus(tag)['rule_a'], chosen_delta(tag)
    margin = '' if delta is None else f'++eval.oneshot.margin_delta={delta}'
    keep = f'{KEEP_DPID}/detections_oneshot_{tag}'
    if len(glob.glob(f'{keep}/*.txt')) != N_VAL:
        shutil.rmtree(VAL_OS, ignore_errors=True)
        !cd /content/pid && python src/run_pipeline.py fgc --infer \
            data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/e2e_dpid_$tag eval.split=val ++fgc.oneshot=true \
            oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.prototypes=$ENC_DIR/$protos oneshot.other_min_cosine=$tau $OS_ARGS \
            2>&1 | grep -v "^Warning" | tail -2
        restore(VAL_OS, keep, N_VAL)
    restore(keep, VAL_OS, N_VAL)
    !cd /content/pid && python src/run_pipeline.py evaluate \
        data_root=$LOCAL_DATA_ROOT run_dir=$RUNS_DIR/eval_wrongbox_$tag eval.mode=det_plus_fgc eval.split=val \
        ++eval.image_subset=val_sel ++fgc.oneshot=true oneshot.encoder_ckpt=$ENC_DIR/$enc oneshot.other_min_cosine=$tau \
        $margin $OS_ARGS 2>&1 | grep -v "^Warning" | grep "oneshot_e2e"
    o = metrics_of(f'eval_wrongbox_{tag}')['oneshot_e2e']; w = o['wrong_boxes']; t = o['true_vocab_boxes']
    print(f"{NAME[tag]} (val_sel, {o['n_images']} ảnh, τ={o['tau']:.4f}, δ={o['margin_delta']}): {o['n_predictions']} dự đoán, "
          f"box sai {w['n']} ({fmt(w['frac_of_predictions'])})  bị loại: τ {fmt(w['rejected_tau'])}"
          + ('' if 'rejected_tau_or_margin' not in w else f", τ|lề {fmt(w['rejected_tau_or_margin'])}"))
    print(f"    box đúng (GT vocab, n={t['n']}) bị loại nhầm: τ {fmt(t['rejected_tau'])}"
          + ('' if 'rejected_tau_or_margin' not in t else f", τ|lề {fmt(t['rejected_tau_or_margin'])}")
          + f"   |  end-to-end val_sel: P {fmt(o['precision'])} R {fmt(o['recall'])} R box {fmt(o['box_recall'])}")

## 13. PHASE C Bước 2B — encoder DINOv2 + A4 + CL2N (không train)

`docs/PLAN_PHASE_C.md`: Bước 1 (cục bộ, tập quyết định **P1-val** trên `val_sel`, luật chốt trước) cho **DINOv2 ViT-S/14**
(tự giám sát, LVD-142M, không nhãn, **không train trên P&ID**) + **A4** (`++oneshot.input_norm=ink_letterbox`: nhị phân Otsu, cắt
sát phần mực của ký hiệu ở tâm, letterbox giữ tỉ lệ — áp **giống hệt** cho ảnh mẫu ISO và mọi crop) + **CL2N** là ĐẠT
(6/10 lớp, Δtop-1 +0.30). Mục này đo nó trên **đủ** P1 và P4 như mọi encoder khác; số mẫu P1 cục bộ (top-1 0.58) chỉ là tham khảo.

- Checkpoint `__dinov2__.pt` dựng bằng chính `scripts/make_imagenet_encoder.py` với `DINO` (backbone, `imgsz=224`, `embed_dim=384`).
  **Mọi lệnh đọc encoder này phải có `$DINO`** (thiếu → lỗi embed_dim / prototype không khớp, code chặn).
- Prototype, tâm CL2N và mọi lượt chạy phải cùng `$A4` (code chặn nếu lệch). Nhóm lớp theo R1a (`groups_ckpt`, như 10b / 10d).
- τ chọn lại trên `val_sel` (quy tắc 9a) với tâm; **không chọn gì trên P1 / OPEN100-ISO**. Báo cạnh bản cũ, không thay thế.

Cần: mục 1–4, phần def đầu cell 9a, 10d (tâm + τ của ImageNet / R1a có CL2N, cho 13c), 11a (OPEN100-ISO, cho 13c).

In [ ]:
# 13a. Encoder DINOv2 (không train) + 42 prototype có A4, rồi separation
import json, shutil, torch
from utils.oneshot import PrototypeDirectory
DINO = 'fgc.backbone=vit_small_patch14_dinov2.lvd142m fgc.imgsz=224 fgc.arcface.embed_dim=384'
A4 = '++oneshot.input_norm=ink_letterbox'
!cd /content/pid && python scripts/make_imagenet_encoder.py --out $ENC_DIR/__dinov2__.pt \
    data_root=$LOCAL_DATA_ROOT $DINO 2>&1 | grep -v "^Warning" | tail -1
st = torch.load(f'{ENC_DIR}/__dinov2__.pt', map_location='cpu', weights_only=False)
assert st['fine_classes'] == [] and st['pretrained_only'] and not st['imagenet_only'], st['pretrained_weights']
print(f"__dinov2__.pt: {st['pretrained_weights']}, head {tuple(st['model']['head.weight'].shape)}")
!cd /content/pid && python src/run_pipeline.py prototypes run_dir=$RUNS_DIR/r2 \
    oneshot.encoder_ckpt=$ENC_DIR/__dinov2__.pt oneshot.prototypes=$ENC_DIR/iso_prototypes_dinov2_a4.pt \
    oneshot.symbols_dir=$REPO_DATA/iso_symbols_clean oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv $DINO $A4 \
    2>&1 | grep -v "^Warning" | grep "oneshot\]"
d = PrototypeDirectory.load(f'{ENC_DIR}/iso_prototypes_dinov2_a4.pt')
assert d.meta['input_norm'] == 'ink_letterbox' and d.meta['imgsz'] == 224 and len(d) == 42, d.meta

In [ ]:
# 13b. Tâm CL2N (crop train lớp R1a đã thấy, trừ oneshot.holdout) → τ trên val_sel → P1 đủ, rồi bảng cạnh R1a / ImageNet+CL2N
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
GROUPS = f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global_lo__.pt'
CENTER_D = f'{ENC_DIR}/center_dinov2_a4.pt'
BASE_D = (f'data_root={LOCAL_DATA_ROOT} oneshot.encoder_ckpt={ENC_DIR}/__dinov2__.pt '
          f'oneshot.prototypes={ENC_DIR}/iso_prototypes_dinov2_a4.pt oneshot.vocab_csv={REPO_DATA}/iso_symbols/vocab.csv '
          f'oneshot.aggregate=mean {DINO} {A4}')
print('===== tâm DINOv2 + A4')
!cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_center_dinov2 eval.mode=oneshot \
    ++eval.oneshot.protocol=center $GROUPS ++eval.oneshot.max_per_class=200 $BASE_D 2>&1 | grep -v "^Warning" | grep "oneshot/center"
c = latest('eval_center_dinov2')
shutil.copy2(c['center_file'], CENTER_D)
assert c['held_out_dropped'] == [] and c['input_norm'] == 'ink_letterbox', (c['held_out_dropped'], c['input_norm'])
print(f"  {c['n_crops']} crop ({c['n_views']} view), {len(c['classes'])} lớp, |tâm|={c['center_norm']:.3f} -> {CENTER_D}")
print('===== τ DINOv2 + A4 + CL2N (val_sel)')
!cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_tau_dinov2 eval.mode=oneshot \
    ++eval.oneshot.protocol=tau ++eval.image_subset=val_sel $GROUPS ++oneshot.center=$CENTER_D $BASE_D 2>&1 | grep -v "^Warning" | grep "oneshot/tau"
o = latest('eval_tau_dinov2')
assert o['classes'] == latest('eval_tau_lo')['classes'] and o['center'] and o['input_norm'] == 'ink_letterbox', o['classes']
print(f"  A tau={o['rule_a']['tau']:.4f}   B tau={o['rule_b']['tau']:.4f} (J={o['rule_b']['youden_j']:.3f})   δ lề={o['margin_rule_a']['delta']:.4f}")
print('===== P1 DINOv2 + A4 + CL2N')
!cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p1_dinov2 eval.mode=oneshot \
    ++eval.oneshot.protocol=p1 ++oneshot.center=$CENTER_D $BASE_D 2>&1 | grep -v "^Warning" | grep "oneshot/p1"

fmt = lambda v: '     -' if v is None else f'{v:6.4f}'
RUNS13 = (('R1a', 'eval_p1', 'lo'), ('ImageNet+CL2N', 'eval_p1_imagenet_cl2n', 'imagenet_cl2n'),
          ('DINOv2+A4+CL2N', 'eval_p1_dinov2', 'dinov2'))
have = [(n, latest(r), t) for n, r, t in RUNS13 if glob.glob(f'{RUNS_DIR}/{r}/*/metrics.json')]
assert len({o['n_crops'] for _, o, _ in have}) == 1, [(n, o['n_crops']) for n, o, _ in have]   # cùng crop P1
assert have[-1][1]['input_norm'] == 'ink_letterbox' and have[-1][1]['center'], 'lượt P1 DINOv2 thiếu A4 / tâm'
print(f"\n{'':18s}" + ''.join(f'{n:>16s}' for n, _, _ in have))
for label, key in (('AUROC (cos)', lambda o: o['open_set']['auroc']), ('AUROC (lề)', lambda o: o['open_set'].get('margin_auroc')),
                   ('OSCR', lambda o: o['open_set']['oscr_area']), ('h_in macro top1', lambda o: o['h_in']['macro_top1']),
                   ('h_in macro top3', lambda o: o['h_in']['macro_top3'])):
    print(f'  {label:16s}' + ''.join(f'{fmt(key(o)):>16s}' for _, o, _ in have))
if 'at_chosen_taus' in globals():
    for n, o, t in have:
        print(f'Tại τ chọn trên val_sel ({n}):'); at_chosen_taus(o, t, {'H_chon'}, {'OOV_heldout'})
print('Từng lớp top1: ' + ' | '.join(n for n, _, _ in have))
for name in have[0][1]['per_class']:
    a = [o['per_class'][name] for _, o, _ in have]
    print(f"  {name:10s} {a[0]['group']:12s}  top1 " + ' | '.join(fmt(x['top1']) for x in a)
          + '   top3 ' + ' | '.join(fmt(x['top3']) for x in a) + '   hay đoán ' + ' | '.join(str(x['most_predicted']) for x in a))

**13c–13d. P4 trên OPEN100-ISO** cho DINOv2 + A4 + CL2N, kèm hai dòng CL2N đã hẹn (PLAN_PHASE_C §4): ImageNet + CL2N, R1a + CL2N
(tâm `center_imagenet.pt` / `center_lo.pt` của 10d). Đúng lệnh §11b, cùng quy tắc báo cáo (suy giảm = số chính, theo tờ). τ / δ của
mỗi dòng lấy từ lượt τ cùng tag (`eval_tau_dinov2`, `eval_tau_imagenet_cl2n`, `eval_tau_lo_cl2n`). Cần 11a (`LOCAL_ISO_ROOT`).

In [ ]:
# 13c. P4: DINOv2 + A4 + CL2N, ImageNet + CL2N, R1a + CL2N (nhóm lớp theo R1a, như §11b)
assert os.path.exists(f'{LOCAL_ISO_ROOT}/canonical/annotations.coco.json'), 'chạy 11a (giải nén OPEN100-ISO) trước'
P4_RUNS13 = (
    ('dinov2', f'oneshot.encoder_ckpt={ENC_DIR}/__dinov2__.pt oneshot.prototypes={ENC_DIR}/iso_prototypes_dinov2_a4.pt '
               f'{DINO} {A4} ++oneshot.center={ENC_DIR}/center_dinov2_a4.pt {GROUPS}'),
    ('imagenet_cl2n', f'oneshot.encoder_ckpt={ENC_DIR}/__imagenet__.pt oneshot.prototypes={ENC_DIR}/iso_prototypes_imagenet.pt '
                      f'++oneshot.center={ENC_DIR}/center_imagenet.pt {GROUPS}'),
    ('lo_cl2n', f'oneshot.encoder_ckpt={ENC_DIR}/__global_lo__.pt oneshot.prototypes={ENC_DIR}/iso_prototypes_lo.pt '
                f'++oneshot.center={ENC_DIR}/center_lo.pt'),
)
for tag, args in P4_RUNS13:
    print(f'===== P4 {tag}')
    !cd /content/pid && python src/run_pipeline.py evaluate \
        data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/eval_p4_$tag eval.mode=oneshot ++eval.oneshot.protocol=p4 \
        oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv oneshot.aggregate=mean $args 2>&1 | grep -v "^Warning" | tail -2

In [ ]:
# 13d. Bảng P4: R1a, ImageNet (§11) cạnh ImageNet+CL2N, R1a+CL2N, DINOv2+A4+CL2N (13c); suy giảm trước, sạch sau
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
TAGS13 = ('lo', 'imagenet', 'lo_cl2n', 'imagenet_cl2n', 'dinov2')
NAME13 = {'lo': 'R1a', 'imagenet': 'ImageNet', 'lo_cl2n': 'R1a+CL2N', 'imagenet_cl2n': 'ImageNet+CL2N', 'dinov2': 'DINOv2+A4+CL2N'}
P4b = {t: latest(f'eval_p4_{t}') for t in TAGS13 if glob.glob(f'{RUNS_DIR}/eval_p4_{t}/*/metrics.json')}
assert len({o['degraded']['open_set']['n_known'] for o in P4b.values()}) == 1, 'các lượt P4 khác tập crop?'
fmt = lambda v: '    -' if v is None else f'{v:5.3f}'
pm = lambda ms: '     -     ' if ms['mean'] is None else f"{ms['mean']:.3f}±{ms['std'] or 0:.3f}"
for view in ('degraded', 'clean'):
    print(f"\n===== {view} ({'SỐ CHÍNH' if view == 'degraded' else 'tham chiếu'})   ô: top1 top3 | top1 theo tờ (số lớp)")
    print(f"  {'':11s}" + ''.join(f'{NAME13[t]:>32s}' for t in P4b))
    for g in ('S', 'V', 'H_chon', 'H_tu_nhien'):
        cells = [f"{fmt(o[view]['groups'][g]['macro_top1'])} {fmt(o[view]['groups'][g]['macro_top3'])} | "
                 f"{pm(o[view]['sheet_mean_std']['groups'][g]['macro_top1'])} ({o[view]['groups'][g]['n_classes']:2d})" for o in P4b.values()]
        print(f'  {g:11s}' + ''.join(f'{c:>32s}' for c in cells))
    for label, key in (('AUROC cos', lambda b: fmt(b['open_set']['auroc'])), ('AUROC lề', lambda b: fmt(b['open_set']['margin_auroc'])),
                       ('OSCR', lambda b: fmt(b['open_set']['oscr_area'])), ('AUROC/tờ', lambda b: pm(b['sheet_mean_std']['auroc'])),
                       ('isa AUROC', lambda b: fmt(b['other']['other_isa']['auroc']))):
        print(f'  {label:11s}' + ''.join(f'{key(o[view]):>32s}' for o in P4b.values()))
if 'at_chosen_taus' in globals():
    for tag in ('lo_cl2n', 'imagenet_cl2n', 'dinov2'):
        if tag in P4b:
            o = P4b[tag]
            print(f'\n{NAME13[tag]} — degraded, tại τ / δ chọn trên val_sel:')
            at_chosen_taus(o, tag, {'S', 'V', 'H_chon', 'V_held', 'H_tu_nhien'}, {'other_iso'},
                           keep=lambda r: r['split'] == 'degraded', open_set=o['degraded']['open_set'])

In [ ]:
# 13e (tuỳ chọn). End-to-end DINOv2 + A4 + CL2N trên detection agnostic_merge — cần cell 12a đã chạy trong phiên này
assert 'restore' in globals() and 'chosen_taus' in globals(), 'chạy cell 12a (DET_CKPT, restore…) và phần def đầu cell 9a'
tag = 'dinov2'
tau, delta = chosen_taus(tag)['rule_a'], chosen_delta(tag)
D_ARGS = (f'oneshot.encoder_ckpt={ENC_DIR}/__dinov2__.pt oneshot.prototypes={ENC_DIR}/iso_prototypes_dinov2_a4.pt '
          f'{DINO} {A4} ++oneshot.center={ENC_DIR}/center_dinov2_a4.pt')
ISO_OS = f'{LOCAL_ISO_ROOT}/derived/detections_oneshot/test'
restore(f'{KEEP_ISO}/detections_agnostic', ISO_DET, 864)
keep = f'{KEEP_ISO}/detections_oneshot_{tag}'
if len(glob.glob(f'{keep}/*.txt')) != 864:
    shutil.rmtree(ISO_OS, ignore_errors=True)
    !cd /content/pid && python src/run_pipeline.py fgc --infer \
        data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/e2e_iso_$tag eval.split=test ++fgc.oneshot=true \
        ++detector.classes_yaml=$DET_CLASSES $D_ARGS oneshot.other_min_cosine=$tau $OS_ARGS 2>&1 | grep -v "^Warning" | tail -2
    restore(ISO_OS, keep, 864)
restore(keep, ISO_OS, 864)
for suffix, margin in (('', ''), ('_margin', f'++eval.oneshot.margin_delta={delta}')):
    print(f'===== e2e DINOv2  τ={tau:.4f}' + (f'  δ={delta:.4f}' if suffix else ''))
    !cd /content/pid && python src/run_pipeline.py evaluate \
        data_root=$LOCAL_ISO_ROOT run_dir=$RUNS_DIR/eval_e2e_iso_{tag}{suffix} eval.mode=det_plus_fgc eval.split=test \
        ++fgc.oneshot=true $D_ARGS $GROUPS oneshot.other_min_cosine=$tau $margin $OS_ARGS 2>&1 | grep -v "^Warning" | grep "oneshot_e2e"
NAME['dinov2'] = 'DINOv2+A4+CL2N'
if not any(t == 'dinov2' for t, *_ in E2E_RUNS):
    E2E_RUNS = E2E_RUNS + (('dinov2', '__dinov2__.pt', 'iso_prototypes_dinov2_a4.pt'),)
print('-> chạy lại cell 12b (bảng) để in cả dòng DINOv2')

## 14. PHASE C §2E.3 — kiểm lại F0 (căn chỉnh ảnh mẫu) trên P1 đủ + P4

`docs/PLAN_PHASE_C.md` §2E.1–2E.3 (luật chốt 2026-11-18, trước khi chạy). Encoder, 42 prototype và tâm CL2N của §13 giữ nguyên
(**không dựng lại**); chỉ ảnh mẫu đổi trong không gian CL2N, truy vấn không đổi:
- `f0a` — `++oneshot.template_align=domain_center`: ảnh mẫu trừ trung bình 42 ảnh mẫu (thay vì tâm crop).
- `shift` — `++oneshot.template_align=pair_shift ++oneshot.template_pairs=…`: cộng phép dịch trung bình (crop − ảnh mẫu) của 10 lớp
  có cặp thật, tính lúc nạp với chính tâm `center_dinov2_a4.pt` (code chặn nếu ảnh mẫu trong file cặp lệch prototype > 1e-3).

Mỗi dòng: τ trên `val_sel` → P1 đủ (14a) → P4 (14b) — lệnh giống hệt 13b / 13c; 14c in bảng nhanh. Không chọn gì trên P1 / P4. **Quyết định chạy cục bộ**
(`scripts/f0_colab_compare.py`, bootstrap) trên CSV mà 14a–14b chép vào `runs/f0_csv/` trên Drive.

Cần: mục 1–4, phần def đầu cell 9a, 11a (OPEN100-ISO), và trong `encoders/` trên Drive: `__dinov2__.pt`, `iso_prototypes_dinov2_a4.pt`,
`center_dinov2_a4.pt` (13a–13b) và **`pairs_dinov2_a4.pt`** (tải lên từ máy: `runs/phase_c/f0/pairs_dinov2_a4.pt`, 3.7 MB).

In [ ]:
# 14a. F0: τ (val_sel) → P1 đủ cho f0a và shift (DINOv2 + A4 + CL2N của §13; chỉ ảnh mẫu đổi)
import glob, json, os, shutil
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
DINO = 'fgc.backbone=vit_small_patch14_dinov2.lvd142m fgc.imgsz=224 fgc.arcface.embed_dim=384'
A4 = '++oneshot.input_norm=ink_letterbox'
GROUPS = f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global_lo__.pt'
CENTER_D = f'{ENC_DIR}/center_dinov2_a4.pt'
PAIRS = f'{ENC_DIR}/pairs_dinov2_a4.pt'
for f in ('__dinov2__.pt', 'iso_prototypes_dinov2_a4.pt', 'center_dinov2_a4.pt', 'pairs_dinov2_a4.pt'):
    assert os.path.exists(f'{ENC_DIR}/{f}'), f'thiếu {ENC_DIR}/{f} (13a–13b; pairs: tải runs/phase_c/f0/pairs_dinov2_a4.pt lên)'
ENC_D = (f'oneshot.encoder_ckpt={ENC_DIR}/__dinov2__.pt oneshot.prototypes={ENC_DIR}/iso_prototypes_dinov2_a4.pt '
         f'oneshot.vocab_csv={REPO_DATA}/iso_symbols/vocab.csv oneshot.aggregate=mean {DINO} {A4} ++oneshot.center={CENTER_D}')
F0 = (('f0a', 'domain_center', ''), ('shift', 'pair_shift', f'++oneshot.template_pairs={PAIRS}'))
CSV_DIR = f'{RUNS_DIR}/f0_csv'
os.makedirs(CSV_DIR, exist_ok=True)
for v, kind, extra in F0:
    tag, ALIGN = f'dinov2_{v}', f'++oneshot.template_align={kind} {extra}'
    print(f'===== {tag}: τ (val_sel)')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_tau_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=tau ++eval.image_subset=val_sel data_root=$LOCAL_DATA_ROOT $GROUPS $ENC_D $ALIGN 2>&1 | grep -v "^Warning" | grep "oneshot/tau"
    print(f'===== {tag}: P1 đủ')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p1_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=p1 data_root=$LOCAL_DATA_ROOT $ENC_D $ALIGN 2>&1 | grep -v "^Warning" | grep "oneshot/p1"
    for p in ('tau', 'p1'):
        o = latest(f'eval_{p}_{tag}')
        assert o['template_align']['kind'] == kind and o['center'] and o['input_norm'] == 'ink_letterbox', (p, o['template_align'])
    assert latest(f'eval_p1_{tag}')['n_crops'] == latest('eval_p1_dinov2')['n_crops'], 'P1 khác tập crop của 2B'
    src = sorted(glob.glob(f'{RUNS_DIR}/eval_p1_{tag}/*/oneshot_p1_crops.csv'), key=os.path.getmtime)[-1]
    shutil.copy2(src, f'{CSV_DIR}/oneshot_p1_crops_dinov2_{v}.csv')
    print('  template_align:', latest(f'eval_p1_{tag}')['template_align'])

In [ ]:
# 14b. F0: P4 trên OPEN100-ISO cho f0a và shift (lệnh như 13c) — cần 14a (biến) và 11a
assert os.path.exists(f'{LOCAL_ISO_ROOT}/canonical/annotations.coco.json'), 'chạy 11a (giải nén OPEN100-ISO) trước'
for v, kind, extra in F0:
    tag, ALIGN = f'dinov2_{v}', f'++oneshot.template_align={kind} {extra}'
    print(f'===== {tag}: P4')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p4_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=p4 data_root=$LOCAL_ISO_ROOT $GROUPS $ENC_D $ALIGN 2>&1 | grep -v "^Warning" | tail -2
    o = latest(f'eval_p4_{tag}')
    assert o['template_align']['kind'] == kind and o['center'] and o['input_norm'] == 'ink_letterbox', o['template_align']
    src = sorted(glob.glob(f'{RUNS_DIR}/eval_p4_{tag}/*/oneshot_p4_crops.csv'), key=os.path.getmtime)[-1]
    shutil.copy2(src, f'{CSV_DIR}/oneshot_p4_crops_dinov2_{v}.csv')

In [ ]:
# 14c. Bảng nhanh cạnh 2B (quyết định: cục bộ, scripts/f0_colab_compare.py trên CSV ở runs/f0_csv/)
fmt = lambda v: '     -' if v is None else f'{v:6.4f}'
ROWS14 = (('2B', 'dinov2'), ('f0a', 'dinov2_f0a'), ('shift', 'dinov2_shift'))
KEYS14 = [('τ A', 'tau', lambda o: o['rule_a']['tau']), ('τ B', 'tau', lambda o: o['rule_b']['tau']),
          ('δ lề', 'tau', lambda o: o['margin_rule_a']['delta']),
          ('P1 h_in top1', 'p1', lambda o: o['h_in']['macro_top1']), ('P1 h_in top3', 'p1', lambda o: o['h_in']['macro_top3']),
          ('P1 AUROC cos', 'p1', lambda o: o['open_set']['auroc']), ('P1 AUROC lề', 'p1', lambda o: o['open_set'].get('margin_auroc')),
          ('P4 AUROC cos', 'p4', lambda o: o['degraded']['open_set']['auroc']),
          ('P4 AUROC lề', 'p4', lambda o: o['degraded']['open_set']['margin_auroc'])]
KEYS14 += [(f'P4 {g} top1', 'p4', lambda o, g=g: o['degraded']['groups'][g]['macro_top1']) for g in ('S', 'V', 'H_chon', 'H_tu_nhien')]
print(f"{'':18s}" + ''.join(f'{n:>10s}' for n, _ in ROWS14))
for label, p, key in KEYS14:
    print(f'  {label:16s}' + ''.join(f'{fmt(key(latest(f"eval_{p}_{t}"))):>10s}' for _, t in ROWS14))
print(f'\nTải về máy, đặt vào runs/: ' + ', '.join(sorted(os.listdir(CSV_DIR))))
print('rồi chạy: python scripts/f0_colab_compare.py --out runs/phase_c/f0_colab')

## 15. PHASE C §2G — Điểm 4: bỏ ★ `pipeline` (41 ảnh mẫu)

`docs/PLAN_PHASE_C.md` §2G (luật chốt 2026-11-19, trước khi chạy). User (chuyên gia P&ID): `pipeline` là nét nối, không phải ký hiệu
→ hệ thống đề xuất bỏ nó: `++oneshot.exclude_prototypes=[pipeline]`. Encoder, 42 prototype và tâm CL2N của §13 giữ nguyên (**không dựng lại**).

- 15a: τ (val_sel, nhóm R1a) cho 41 → **P2** (val, `test`, nhóm R1b `__global__.pt`) cho cả 42 và 41 — 2B chưa từng chạy P2, đây là
  bằng chứng sạch → P1 đủ cho 41 (**hậu kiểm**).
- 15b: P4 cho 41 (**hậu kiểm**). 15c: bảng nhanh.

Không chọn gì ở đây. **Áp luật cục bộ** (`scripts/nopipe_compare.py`: kiểm 41 = 42 bỏ `pipeline`, bootstrap) trên CSV mà 15a–15b
chép vào `runs/nopipe_csv/` trên Drive.

Cần: mục 1–4, 11a (OPEN100-ISO), và trong `encoders/` trên Drive: `__dinov2__.pt`, `iso_prototypes_dinov2_a4.pt`, `center_dinov2_a4.pt`
(13a–13b), `__global__.pt`, `__global_lo__.pt`.

In [ ]:
# 15a. Điểm 4 (PLAN_PHASE_C §2G): 2B với 41 ảnh mẫu (bỏ pipeline) — τ (val_sel) → P2 (42 và 41) → P1 đủ (41, hậu kiểm)
import glob, json, os, shutil
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
newest_csv = lambda run, p: sorted(glob.glob(f'{RUNS_DIR}/{run}/*/oneshot_{p}_crops.csv'), key=os.path.getmtime)[-1]
DINO = 'fgc.backbone=vit_small_patch14_dinov2.lvd142m fgc.imgsz=224 fgc.arcface.embed_dim=384'
A4 = '++oneshot.input_norm=ink_letterbox'
GROUPS = f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global_lo__.pt'     # τ, P4: nhóm R1a (như §13)
GROUPS_P2 = f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global__.pt'     # P2: nhóm R1b (như §9)
CENTER_D = f'{ENC_DIR}/center_dinov2_a4.pt'
for f in ('__dinov2__.pt', 'iso_prototypes_dinov2_a4.pt', 'center_dinov2_a4.pt', '__global__.pt', '__global_lo__.pt'):
    assert os.path.exists(f'{ENC_DIR}/{f}'), f'thiếu {ENC_DIR}/{f}'
ENC_D = (f'oneshot.encoder_ckpt={ENC_DIR}/__dinov2__.pt oneshot.prototypes={ENC_DIR}/iso_prototypes_dinov2_a4.pt '
         f'oneshot.vocab_csv={REPO_DATA}/iso_symbols/vocab.csv oneshot.aggregate=mean {DINO} {A4} ++oneshot.center={CENTER_D}')
NOPIPE = "'++oneshot.exclude_prototypes=[pipeline]'"
CSV_DIR = f'{RUNS_DIR}/nopipe_csv'
os.makedirs(CSV_DIR, exist_ok=True)
print('===== dinov2_nopipe: τ (val_sel)')
!cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_tau_dinov2_nopipe eval.mode=oneshot \
    ++eval.oneshot.protocol=tau ++eval.image_subset=val_sel data_root=$LOCAL_DATA_ROOT $GROUPS $ENC_D $NOPIPE 2>&1 | grep -v "^Warning" | grep "oneshot/tau"
for tag, extra in (('dinov2', ''), ('dinov2_nopipe', NOPIPE)):
    print(f'===== {tag}: P2 (val, test)')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p2_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=p2 ++eval.image_subset=test data_root=$LOCAL_DATA_ROOT $GROUPS_P2 $ENC_D $extra 2>&1 | grep -v "^Warning" | grep "oneshot/p2"
print('===== dinov2_nopipe: P1 đủ (hậu kiểm)')
!cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p1_dinov2_nopipe eval.mode=oneshot \
    ++eval.oneshot.protocol=p1 data_root=$LOCAL_DATA_ROOT $ENC_D $NOPIPE 2>&1 | grep -v "^Warning" | grep "oneshot/p1"
for p, tag, n in (('tau', 'dinov2_nopipe', 41), ('p2', 'dinov2', 42), ('p2', 'dinov2_nopipe', 41), ('p1', 'dinov2_nopipe', 41)):
    o = latest(f'eval_{p}_{tag}')
    assert o['n_prototypes'] == n and o['center'] and o['input_norm'] == 'ink_letterbox', (p, tag, o['n_prototypes'])
assert latest('eval_p2_dinov2')['n_crops'] == latest('eval_p2_dinov2_nopipe')['n_crops'], 'P2 42 / 41 khác tập crop'
assert latest('eval_p1_dinov2_nopipe')['n_crops'] == latest('eval_p1_dinov2')['n_crops'], 'P1 khác tập crop của 2B'
for p, tag in (('p2', 'dinov2'), ('p2', 'dinov2_nopipe'), ('p1', 'dinov2_nopipe')):
    shutil.copy2(newest_csv(f'eval_{p}_{tag}', p), f'{CSV_DIR}/oneshot_{p}_crops_{tag}.csv')
print('đã chép:', sorted(os.listdir(CSV_DIR)))

In [ ]:
# 15b. Điểm 4: P4 trên OPEN100-ISO với 41 ảnh mẫu (hậu kiểm; lệnh như 13c) — cần 15a (biến) và 11a
assert os.path.exists(f'{LOCAL_ISO_ROOT}/canonical/annotations.coco.json'), 'chạy 11a (giải nén OPEN100-ISO) trước'
print('===== dinov2_nopipe: P4')
!cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p4_dinov2_nopipe eval.mode=oneshot \
    ++eval.oneshot.protocol=p4 data_root=$LOCAL_ISO_ROOT $GROUPS $ENC_D $NOPIPE 2>&1 | grep -v "^Warning" | tail -2
o = latest('eval_p4_dinov2_nopipe')
assert o['n_prototypes'] == 41 and o['center'] and o['input_norm'] == 'ink_letterbox', o['n_prototypes']
shutil.copy2(newest_csv('eval_p4_dinov2_nopipe', 'p4'), f'{CSV_DIR}/oneshot_p4_crops_dinov2_nopipe.csv')
print('đã chép:', sorted(os.listdir(CSV_DIR)))

In [ ]:
# 15c. Bảng nhanh 42 / 41 (áp luật §2G: cục bộ, scripts/nopipe_compare.py trên CSV ở runs/nopipe_csv/)
fmt = lambda v: '     -' if v is None else f'{v:6.4f}'
ROWS15 = (('42 (2B)', 'dinov2'), ('41', 'dinov2_nopipe'))
KEYS15 = [('τ A', 'tau', lambda o: o['rule_a']['tau']), ('δ lề', 'tau', lambda o: o['margin_rule_a']['delta']),
          ('P2 S top1', 'p2', lambda o: o['groups']['S']['macro_top1']), ('P2 V top1', 'p2', lambda o: o['groups']['V']['macro_top1']),
          ('P2 AUROC cos', 'p2', lambda o: o['open_set']['auroc']), ('P2 AUROC lề', 'p2', lambda o: o['open_set'].get('margin_auroc')),
          ('P1 h_in top1 *', 'p1', lambda o: o['h_in']['macro_top1']), ('P1 AUROC cos *', 'p1', lambda o: o['open_set']['auroc']),
          ('P4 AUROC cos *', 'p4', lambda o: o['degraded']['open_set']['auroc'])]
KEYS15 += [(f'P4 {g} top1 *', 'p4', lambda o, g=g: o['degraded']['groups'][g]['macro_top1']) for g in ('S', 'V', 'H_chon', 'H_tu_nhien')]
print(f"{'':18s}" + ''.join(f'{n:>10s}' for n, _ in ROWS15) + '      (* = hậu kiểm)')
for label, p, key in KEYS15:
    print(f'  {label:16s}' + ''.join(f'{fmt(key(latest(f"eval_{p}_{t}"))):>10s}' for _, t in ROWS15))
print(f'\nTải về máy, đặt vào runs/: ' + ', '.join(sorted(os.listdir(CSV_DIR))))
print('rồi chạy: python scripts/nopipe_compare.py --out runs/phase_c/nopipe')

## 16. PHASE C §2K.6 — (i) dạng B của butterfly + (ii) gộp view `max`

`docs/PLAN_PHASE_C.md` §2K.4–2K.6 (luật chốt 2026-11-20, trước khi chạy). Đã chọn trên **P4-val** (bộ render OPEN100-ISO trên khung
xương khác, cục bộ); ở đây **chỉ xác nhận**. Mốc = 2B 41 ảnh mẫu, gộp trung bình = **§15** (`dinov2_nopipe`) → **chạy §15 trước**.
Encoder, tâm CL2N của §13 giữ nguyên; mọi dòng bỏ `pipeline`. Ba dòng:
- `formB` — 43 prototype (+ `valve_butterfly@B`, ảnh `data/raw/iso_symbols_forms/` đi qua git), gộp trung bình;
- `max` — 42 prototype, `oneshot.aggregate=max`;
- `formB_max` — cả hai.

16a dựng file 43 prototype (+ kiểm 42 hàng đầu trùng file của 13a). 16b: mỗi dòng τ (val_sel, nhóm R1a) → P2 (val, `test`, nhóm R1b)
→ P1 đủ. 16c: P4. 16d: bảng nhanh. Không chọn gì ở đây. **Áp luật cục bộ** (`scripts/k4_colab_compare.py`: kiểm nhất quán, bootstrap)
trên CSV mà 16b–16c chép vào `runs/k4_csv/` (cùng CSV mốc ở `runs/nopipe_csv/`).

Cần: mục 1–4, 11a (OPEN100-ISO), §15 đã chạy, và trong `encoders/` trên Drive: `__dinov2__.pt`, `iso_prototypes_dinov2_a4.pt`,
`center_dinov2_a4.pt`, `__global__.pt`, `__global_lo__.pt`.


In [ ]:
# 16a. §2K.6: dựng 43 prototype (+ dạng B của butterfly) bằng encoder 2B, kiểm 42 hàng đầu trùng file của 13a
import glob, json, os, shutil, torch
from utils.oneshot import PrototypeDirectory
latest = lambda run: json.load(open(sorted(glob.glob(f'{RUNS_DIR}/{run}/*/metrics.json'), key=os.path.getmtime)[-1]))['oneshot']
newest_csv = lambda run, p: sorted(glob.glob(f'{RUNS_DIR}/{run}/*/oneshot_{p}_crops.csv'), key=os.path.getmtime)[-1]
DINO = 'fgc.backbone=vit_small_patch14_dinov2.lvd142m fgc.imgsz=224 fgc.arcface.embed_dim=384'
A4 = '++oneshot.input_norm=ink_letterbox'
GROUPS = f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global_lo__.pt'     # τ, P4: nhóm R1a (như §13 / §15)
GROUPS_P2 = f'++eval.oneshot.groups_ckpt={ENC_DIR}/__global__.pt'     # P2: nhóm R1b (như §15)
CENTER_D = f'{ENC_DIR}/center_dinov2_a4.pt'
PROTO42 = f'{ENC_DIR}/iso_prototypes_dinov2_a4.pt'
PROTO43 = f'{ENC_DIR}/iso_prototypes_dinov2_a4_formB.pt'
FORMS_DIR = f'{REPO_DATA}/iso_symbols_forms'
for f in ('__dinov2__.pt', 'iso_prototypes_dinov2_a4.pt', 'center_dinov2_a4.pt', '__global__.pt', '__global_lo__.pt'):
    assert os.path.exists(f'{ENC_DIR}/{f}'), f'thiếu {ENC_DIR}/{f}'
assert os.path.exists(f'{FORMS_DIR}/valve_butterfly@B.png'), 'thiếu ảnh dạng B: git pull (data/raw/iso_symbols_forms/)'
assert glob.glob(f'{RUNS_DIR}/eval_p4_dinov2_nopipe/*/metrics.json'), 'chạy §15 (mốc 41) trước'
!cd /content/pid && python src/run_pipeline.py prototypes run_dir=$RUNS_DIR/r2_formB \
    oneshot.encoder_ckpt=$ENC_DIR/__dinov2__.pt oneshot.prototypes=$PROTO43 \
    oneshot.symbols_dir=$REPO_DATA/iso_symbols_clean oneshot.vocab_csv=$REPO_DATA/iso_symbols/vocab.csv $DINO $A4 \
    ++oneshot.extra_forms_dir=$FORMS_DIR 2>&1 | grep -v "^Warning" | grep "oneshot\]"
d42, d43 = PrototypeDirectory.load(PROTO42), PrototypeDirectory.load(PROTO43)
diff = float((d43.embeddings[:42] - d42.embeddings).abs().max())
assert len(d43) == 43 and d43.names[:42] == d42.names and diff <= 1e-5, (len(d43), diff)
assert d43.meta['extra_forms'] == ['valve_butterfly@B'] and d43.meta['input_norm'] == 'ink_letterbox', d43.meta
print(f'43 prototype OK: 42 hàng đầu lệch {diff:.1e}, thêm {d43.meta["extra_forms"]}')


In [ ]:
# 16b. §2K.6: τ (val_sel) → P2 (val, test) → P1 đủ cho formB / max / formB_max — cần 16a
NOPIPE = "'++oneshot.exclude_prototypes=[pipeline]'"
BASE_K4 = (f'oneshot.encoder_ckpt={ENC_DIR}/__dinov2__.pt oneshot.vocab_csv={REPO_DATA}/iso_symbols/vocab.csv '
           f'{DINO} {A4} ++oneshot.center={CENTER_D} {NOPIPE}')
K4 = (('formB', PROTO43, 'mean', 42), ('max', PROTO42, 'max', 41), ('formB_max', PROTO43, 'max', 42))
CSV_DIR = f'{RUNS_DIR}/k4_csv'
os.makedirs(CSV_DIR, exist_ok=True)
for tag, proto, agg, n in K4:
    ENC_K = f'{BASE_K4} oneshot.prototypes={proto} oneshot.aggregate={agg}'
    print(f'===== dinov2_{tag}: τ (val_sel)')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_tau_dinov2_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=tau ++eval.image_subset=val_sel data_root=$LOCAL_DATA_ROOT $GROUPS $ENC_K 2>&1 | grep -v "^Warning" | grep "oneshot/tau"
    print(f'===== dinov2_{tag}: P2 (val, test)')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p2_dinov2_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=p2 ++eval.image_subset=test data_root=$LOCAL_DATA_ROOT $GROUPS_P2 $ENC_K 2>&1 | grep -v "^Warning" | grep "oneshot/p2"
    print(f'===== dinov2_{tag}: P1 đủ')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p1_dinov2_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=p1 data_root=$LOCAL_DATA_ROOT $ENC_K 2>&1 | grep -v "^Warning" | grep "oneshot/p1"
    for p in ('tau', 'p2', 'p1'):
        o = latest(f'eval_{p}_dinov2_{tag}')
        assert o['n_prototypes'] == n and o['aggregate'] == agg and o['center'] and o['input_norm'] == 'ink_letterbox', (p, tag, o['n_prototypes'], o['aggregate'])
    assert latest(f'eval_p1_dinov2_{tag}')['n_crops'] == latest('eval_p1_dinov2_nopipe')['n_crops'], 'P1 khác tập crop của mốc'
    for p in ('p2', 'p1'):
        shutil.copy2(newest_csv(f'eval_{p}_dinov2_{tag}', p), f'{CSV_DIR}/oneshot_{p}_crops_dinov2_{tag}.csv')
print('đã chép:', sorted(os.listdir(CSV_DIR)))


In [ ]:
# 16c. §2K.6: P4 trên OPEN100-ISO cho formB / max / formB_max (lệnh như 15b) — cần 16b (biến) và 11a
assert os.path.exists(f'{LOCAL_ISO_ROOT}/canonical/annotations.coco.json'), 'chạy 11a (giải nén OPEN100-ISO) trước'
for tag, proto, agg, n in K4:
    ENC_K = f'{BASE_K4} oneshot.prototypes={proto} oneshot.aggregate={agg}'
    print(f'===== dinov2_{tag}: P4')
    !cd /content/pid && python src/run_pipeline.py evaluate run_dir=$RUNS_DIR/eval_p4_dinov2_$tag eval.mode=oneshot \
        ++eval.oneshot.protocol=p4 data_root=$LOCAL_ISO_ROOT $GROUPS $ENC_K 2>&1 | grep -v "^Warning" | tail -2
    o = latest(f'eval_p4_dinov2_{tag}')
    assert o['n_prototypes'] == n and o['aggregate'] == agg and o['center'], (tag, o['n_prototypes'], o['aggregate'])
    shutil.copy2(newest_csv(f'eval_p4_dinov2_{tag}', 'p4'), f'{CSV_DIR}/oneshot_p4_crops_dinov2_{tag}.csv')
print('đã chép:', sorted(os.listdir(CSV_DIR)))


In [ ]:
# 16d. Bảng nhanh (áp luật §2K.6: cục bộ, scripts/k4_colab_compare.py trên CSV ở runs/k4_csv/ + runs/nopipe_csv/)
fmt = lambda v: '     -' if v is None else f'{v:6.4f}'
ROWS16 = (('base41', 'dinov2_nopipe'), ('formB', 'dinov2_formB'), ('max', 'dinov2_max'), ('formB_max', 'dinov2_formB_max'))
KEYS16 = [('τ A', 'tau', lambda o: o['rule_a']['tau']), ('δ lề', 'tau', lambda o: o['margin_rule_a']['delta']),
          ('P2 S top1', 'p2', lambda o: o['groups']['S']['macro_top1']), ('P2 V top1', 'p2', lambda o: o['groups']['V']['macro_top1']),
          ('P2 AUROC cos', 'p2', lambda o: o['open_set']['auroc']), ('P1 h_in top1', 'p1', lambda o: o['h_in']['macro_top1']),
          ('P1 AUROC cos', 'p1', lambda o: o['open_set']['auroc']), ('P4 AUROC cos', 'p4', lambda o: o['degraded']['open_set']['auroc'])]
KEYS16 += [(f'P4 {g} top1', 'p4', lambda o, g=g: o['degraded']['groups'][g]['macro_top1']) for g in ('S', 'V', 'H_chon', 'H_tu_nhien')]
print(f"{'':16s}" + ''.join(f'{n:>11s}' for n, _ in ROWS16))
for label, p, key in KEYS16:
    print(f'  {label:14s}' + ''.join(f'{fmt(key(latest(f"eval_{p}_{t}"))):>11s}' for _, t in ROWS16))
print(f'\nTải về máy, đặt vào runs/: ' + ', '.join(sorted(os.listdir(CSV_DIR))) + ' (+ 3 CSV dinov2_nopipe của §15)')
print('rồi chạy: python scripts/k4_colab_compare.py --out runs/phase_c/k4_colab')
